# Paper 1: Lesion-Aware, Clutter-Robust Distillation — full pipeline

This notebook runs every experiment in the methodology document (sections 3–5 and the coding guide in section 7), from the raw datasets to the paper tables, figures and the int8 phone model.

**How to run it on Kaggle (free GPU, about 30 hours per week):**
1. Kaggle → *Datasets* → *New dataset*: upload `plantsegv3.zip` and `PlantDoc-Dataset-master.zip` (Kaggle unzips them). Keep it private.
2. *Code* → *New notebook* → *File* → *Import notebook*: pick this `.ipynb`.
3. Right panel: *Add input* → your dataset. *Settings*: **Accelerator = GPU T4 x2**, **Internet = on** (phone verification needed once).
4. *Save version* → *Save & Run All (Commit)*. A session runs up to 12 hours; training pauses safely at `TIME_BUDGET_H` and saves everything to the output.
5. **To continue:** open the notebook again → *Add input* → *Notebook output files* → the previous version of this notebook → *Save & Run All* again. Cell 1 copies the previous `paper1_runs` folder back and every run resumes where it stopped. Repeat until cell 7 prints `FINISHED`.

The same notebook also runs on Colab (data from Google Drive) and on the lab PC (Windows paths), and it detects which one it is on.

**Run it in three steps (set `RUN_MODE` in cell 1):**

| Step | RUN_MODE | What | Lab PC (1080 Ti), rough |
|---|---|---|---|
| 1 | `sanity` | 64 images, 2 epochs: paths, packages, every code path | ~30 min |
| 2 | `pilot` | 1,600/400/400 images, 256 px, 20 epochs, MobileNetV3, seed 0; trains **both teachers** (SegFormer-B4, ResNet-50) and all methods | overnight |
| 3 | `full` | paper runs: 320 px, 40 epochs + early stopping, trimmed grid, 3 seeds; set `FULL_TEACHER` to the pilot's winner | ~2–4 days |

Go from pilot to full only if the pilot's `GATE_full.txt` says PASS (A7 ≥ best baseline + 1.5 mIoU). Each mode writes to its own folder.

**What runs in `full`, in order** (job queue in `p1lib.py`, with the automatic checks):

| Phase | Runs | Check before it |
|---|---|---|
| P1 | teacher (`FULL_TEACHER`), student-alone A0 (MobileNetV3 × 3 seeds), direct small models SegFormer-B0 and DeepLabV3+ MobileNetV2 (3 seeds) | – |
| P2 | MobileNetV3, seed 0: logit KD, AT, CWD, BPKD, SARD, C-aug + A5 LSD, A6 CPD, A7 full | teacher ≥ best small model + 5 mIoU |
| P3 | the same, seeds 1 and 2; EfficientNet-Lite0: A0, best baseline, A7 (3 seeds) | A7 ≥ best baseline + 1.5 mIoU (seed 0) |
| P4 | ablations A1–A4, MobileNetV3, 3 seeds | – |
| P5 | sensitivity: τ ∈ {1, 2}, λ_lsd ∈ {0.5, 2}, λ_cpd ∈ {0.5, 2} (A7, MobileNetV3, seed 0) | – |

Speed settings: validation every 2 epochs, a run stops early after 8 epochs without improvement, images cached at 2× training size, several data-loading processes (training runs in a worker process, also on Windows). SKD is kept in the code but left out of the grid.

Every finished run is evaluated straight away on the PlantSeg test split, the three frozen clutter levels and PlantDoc-Seg (once your masks exist). The test split is never used for choosing anything.

**If cell 1 stops with "No GPU visible":** the notebook is not running in your GPU environment. In Jupyter: *Kernel → Change kernel → cropai*. If torch still shows `+cpu`, in Anaconda Prompt run `conda activate cropai` then
`pip install --force-reinstall torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu118`, restart the kernel and run all again. The install step in cell 1 pins the installed PyTorch, so it can no longer replace a CUDA build.

**Memory safety:** each training run checks system RAM. Above 88 % it halves the data-loading processes for the next epoch; above 95 % it pauses (frees memory, waits up to a minute). RAM, CPU and GPU use are printed in cell 5 every 2 minutes and stored in every run's `log.csv`.

**Safety while you are away:** every file the notebook writes (checkpoints, results, splits, caches) is written to a temporary file and renamed, so a power cut or forced shutdown never leaves a broken file, and any half-finished result from an older version is detected and redone automatically. You never need to delete files by hand: just run all again. Windows sleep is blocked while it runs (the screen may still turn off). If the GPU reaches 85 °C, training pauses until it cools to 75 °C; temperatures are shown in cell 5 and stored in each run's `log.csv`.

## 0. Pipeline library
Writes `p1lib.py` (data, clutter, models, losses, training queue, metrics). Run this cell first.

In [ ]:
%%writefile p1lib.py
"""
p1lib.py - Lesion-Aware, Clutter-Robust Distillation (Paper 1): full experiment pipeline.

Follows "Paper 1 Methodology" sections 3-5 and 7:
  data      leak-free 70/15/15 split stratified by disease (pHash groups never span splits), cached images
  clutter   leaf-like occluders, shadows, dirt, glare, background swap, blur, JPEG; altered share tracked
  models    EfficientNet-B4 U-Net teacher; MobileNetV3-Small / EfficientNet-Lite0 U-Net students;
            direct small baselines SegFormer-B0 and DeepLabV3+ MobileNetV2 (no distillation)
  losses    LSD, CPD, logit KD, AT, CWD, SKD (pixel + pair-wise), BPKD (body + edge), SARD (re-implemented), C-aug
  runner    resumable job queue with the week-3 and week-7 gates, time budget per session, one worker per GPU
  eval      mIoU, Dice, lesion F1, recall by lesion size, boundary F1, severity error, per disease, clutter, natural clutter

Run as a worker:  python p1lib.py worker <gpu_index>   (the notebook launches these)
"""
import os, sys, json, math, time, random, collections, shutil, glob, traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2

cv2.setNumThreads(0)
import torch, torch.nn as nn, torch.nn.functional as F

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)

DEFAULT_CFG = dict(
    img_size=320, cache_long_side=640,               # cache = 2x train size (enough for random scale up to 2.0)
    # teacher candidates; the pilot trains both and "auto" keeps the one with the higher val mIoU
    teachers={"segformer_b4": dict(arch="Segformer", encoder="mit_b4", transformer=True),
              "resnet50": dict(arch="Unet", encoder="resnet50", transformer=False)},
    teachers_to_train=["segformer_b4", "resnet50"], teacher_choice="auto",
    students=["tu-mobilenetv3_small_100"],              # primary student: every method, ablation and sensitivity run
    secondary_students=["tu-tf_efficientnet_lite0"],    # second student: only A0, best baseline and A7 (shows generality)
    secondary_methods=["A0_student", "best_baseline", "A7_full"],
    direct=[dict(name="D-segformer_b0", arch="Segformer", encoder="mit_b0", transformer=True),
            dict(name="D-dlv3p_mobilenetv2", arch="DeepLabV3Plus", encoder="mobilenet_v2", transformer=False)],
    teacher_epochs=40, student_epochs=40, teacher_bs=8, student_bs=16,
    val_every=2, patience_evals=4,                   # validate every 2 epochs; stop a run after 4 checks (8 epochs) without improvement
    lr_teacher=3e-4, lr_student=1e-3, lr_transformer=6e-5, weight_decay=1e-2, warmup_iters=1500,
    seeds=[0, 1, 2], final_seeds=[],
    kd=dict(tau=4.0, lambda_lsd=1.0, lambda_cpd=1.0, mu_feat=1.0, rho_att=1.0, cpd_kl=1.0,
            alpha_boundary=2.0, beta_size=1.0, gamma_conf=1.0, delta_leaf=0.3,
            feat_stages=[3, 4, 5], clutter_prob=0.5, boundary_k=5, skd_w=1.0),
    size_bins=[0.005, 0.05],          # lesion component area / image area: small < 0.5% <= medium <= 5% < large
    split=[0.70, 0.15, 0.15], phash_thr=6,
    gates=dict(teacher_margin=0.05, full_margin=0.015, stop_on_fail=True),
    pretrained=True, num_workers=2, eval_bs=16,
    quick=False, quick_n=[64, 32, 32], quick_epochs=2, max_phase=99,
    sensitivity=["S_tau1", "S_tau2", "S_lsd0.5", "S_lsd2", "S_cpd0.5", "S_cpd2"],
    time_budget_h=11.0, deadline=None,
    ram_high=88, ram_critical=95,             # % of system RAM: >high halves the data workers next epoch, >critical pauses briefly
    gpu_temp_high=85, gpu_temp_resume=75,     # deg C: pause training at >= high, resume when <= resume (max 30 min per pause)
)

# --------------------------------------------------------------------------------------- config & paths
def load_cfg(work):
    with open(Path(work) / "config.json") as f:
        return json.load(f)


def save_cfg(cfg, work):
    Path(work).mkdir(parents=True, exist_ok=True)
    atomic_write_text(Path(work) / "config.json", json.dumps(cfg, indent=2))


def wp(cfg, *parts):
    return Path(cfg["paths"]["work"]).joinpath(*parts)


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def short(enc):
    return enc.replace("tu-", "").split("/")[-1]


def parse_run(name):
    model, method, seed = name.split("__")
    return model, method, int(seed[1:])


def log(msg):
    print(time.strftime("%H:%M:%S"), msg, flush=True)

def atomic_write_text(path, text):
    """Write to a temporary file, then rename: a crash or power cut never leaves a half-written file."""
    path = Path(path); tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
    os.replace(tmp, path)

def atomic_to_csv(df, path, **kw):
    path = Path(path); tmp = path.with_name(path.name + ".tmp")
    df.to_csv(tmp, **kw); os.replace(tmp, path)

_NVSMI = {"path": None, "checked": False}

def gpu_temp():
    """Hottest GPU temperature in deg C from nvidia-smi, or None if it cannot be read."""
    if not _NVSMI["checked"]:
        _NVSMI["path"] = shutil.which("nvidia-smi"); _NVSMI["checked"] = True
    if not _NVSMI["path"]:
        return None
    try:
        import subprocess
        out = subprocess.run([_NVSMI["path"], "--query-gpu=temperature.gpu", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10).stdout
        vals = [int(x) for x in out.split() if x.strip().isdigit()]
        return max(vals) if vals else None
    except Exception:
        return None

def temp_guard(cfg, name):
    """If the GPU is too hot, stop feeding it work until it cools down. Returns seconds paused."""
    t = gpu_temp(); hi, lo = cfg.get("gpu_temp_high", 85), cfg.get("gpu_temp_resume", 75)
    if t is None or t < hi:
        return 0
    log(f"[{name}] GPU {t} C >= {hi} C: pausing until <= {lo} C")
    t0 = time.time()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    while time.time() - t0 < 1800:
        time.sleep(15)
        t = gpu_temp()
        if t is None or t <= lo:
            break
    log(f"[{name}] resumed after {time.time() - t0:.0f} s (GPU {t} C)")
    return time.time() - t0

def keep_awake():
    """Windows: stop the PC from going to sleep while this process runs (screen may still turn off / lock)."""
    if os.name == "nt":
        try:
            import ctypes
            ctypes.windll.kernel32.SetThreadExecutionState(0x80000000 | 0x00000001 | 0x00000002)   # CONTINUOUS | SYSTEM | DISPLAY: no sleep, screen stays on
            return True
        except Exception:
            return False
    return False

def sys_stats():
    """RAM %, free RAM (GB), CPU %, GPU memory reserved by this process (GB)."""
    try:
        import psutil
        vm = psutil.virtual_memory(); ram, free, cpu = vm.percent, vm.available / 1e9, psutil.cpu_percent(interval=None)
    except Exception:
        ram, free, cpu = float("nan"), float("nan"), float("nan")
    gpu = torch.cuda.memory_reserved() / 1e9 if torch.cuda.is_available() else 0.0
    return dict(ram_pct=round(ram, 1), ram_free_gb=round(free, 2), cpu_pct=round(cpu, 1), gpu_mem_gb=round(gpu, 2))

def ram_guard(cfg, name):
    """If system RAM is nearly full, free caches and wait (up to 60 s) instead of letting Windows swap or crash."""
    st = sys_stats()
    if not (st["ram_pct"] >= cfg.get("ram_critical", 95)):
        return False
    import gc
    log(f"[{name}] RAM {st['ram_pct']}% (free {st['ram_free_gb']} GB): pausing to let memory recover")
    for _ in range(12):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        time.sleep(5)
        if sys_stats()["ram_pct"] < cfg.get("ram_critical", 95) - 3:
            break
    return True

# --------------------------------------------------------------------------------------- data preparation
def find_dataset_root(base, need):
    """First folder under `base` whose sub-folder names cover every alternative group in `need`."""
    base = Path(base)
    if not base.exists():
        return None
    for root, dirs, _ in os.walk(base):
        names = {d.lower() for d in dirs}
        if all(any(a in names for a in alts) for alts in need):
            return Path(root)
    return None


def index_pairs(root):
    """Pairs every image with its mask by base name (never by file name with extension)."""
    root = Path(root)
    img_dir = find_dataset_root(root, [("images", "image", "imgs")])
    img_dir = next(p for p in img_dir.iterdir() if p.is_dir() and p.name.lower() in ("images", "image", "imgs"))
    msk_dir = next(p for p in img_dir.parent.iterdir()
                   if p.is_dir() and p.name.lower() in ("annotations", "annotation", "masks", "mask", "labels"))
    masks = {}
    for p in msk_dir.rglob("*"):
        if p.suffix.lower() in IMG_EXT:
            sp = p.parent.name.lower() if p.parent != msk_dir else "all"
            masks[(sp, p.stem)] = p
            masks.setdefault(("any", p.stem), p)
    rows, missing = [], []
    for p in sorted(img_dir.rglob("*")):
        if p.suffix.lower() not in IMG_EXT:
            continue
        sp = p.parent.name.lower() if p.parent != img_dir else "all"
        m = masks.get((sp, p.stem)) or masks.get(("any", p.stem))
        if m is None:
            missing.append(str(p)); continue
        rows.append(dict(image=str(p), mask=str(m), official_split=sp))
    df = pd.DataFrame(rows)
    df["uid"] = np.arange(len(df))
    return df, missing, img_dir, msk_dir


def leaf_proxy(img):
    """Rough leaf mask from excess-green + Otsu (PlantSeg has no leaf masks; describe it as a proxy)."""
    f = img.astype(np.float32) / 255.0
    exg = 2 * f[..., 1] - f[..., 0] - f[..., 2]
    exg = cv2.normalize(exg, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, leaf = cv2.threshold(exg, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return leaf.astype(np.float32)


def _cache_one(args):
    """Reads one original pair, writes the cached copy (long side = L), returns mask statistics."""
    image, mask, cimg, cmsk, L, size = args
    try:
        need = not (os.path.exists(cimg) and os.path.exists(cmsk))
        img = None if need else cv2.imread(cimg)
        m = None if need else cv2.imread(cmsk, cv2.IMREAD_GRAYSCALE)
        if img is None or m is None:                      # missing or damaged cache (e.g. power cut): rebuild
            need = True
            img = cv2.imread(image); m = cv2.imread(mask, cv2.IMREAD_GRAYSCALE)
        if img is None or m is None:
            return None
        if m.shape != img.shape[:2]:
            m = cv2.resize(m, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        if need:
            s = L / max(img.shape[:2])
            if s < 1:
                img = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
                m = cv2.resize(m, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
            ti, tm = cimg[:-4] + ".tmp.jpg", cmsk[:-4] + ".tmp.png"
            cv2.imwrite(ti, img, [cv2.IMWRITE_JPEG_QUALITY, 95]); cv2.imwrite(tm, m)
            os.replace(ti, cimg); os.replace(tm, cmsk)
        vals, cnt = np.unique(m, return_counts=True)
        fg = vals != 0
        disease = int(vals[fg][np.argmax(cnt[fg])]) if fg.any() else 0
        small = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (size, size), interpolation=cv2.INTER_AREA)
        leaf = leaf_proxy(small)
        n_leaf = cv2.connectedComponentsWithStats(leaf.astype(np.uint8), connectivity=8)[2][1:, 4]
        n_leaves = int((n_leaf > 0.01 * size * size).sum())
        bm = (m > 0).astype(np.uint8)
        n, _, st, _ = cv2.connectedComponentsWithStats(bm, connectivity=8)
        comp = (st[1:, 4] / bm.size).astype(np.float32) if n > 1 else np.zeros(0, np.float32)
        return dict(disease=disease, lesion_frac=float(bm.mean()), bg_share=float(1 - leaf.mean()),
                    n_leaves=n_leaves, n_lesions=int(n - 1), comp_fracs=comp.tolist())
    except Exception:
        return None


def _phash(path):
    import imagehash
    from PIL import Image
    return int(str(imagehash.phash(Image.open(path).convert("RGB"))), 16)


def group_near_duplicates(hashes, thr):
    """Union-find over pairs with pHash Hamming distance <= thr. Returns a group id per image."""
    H = np.array(hashes, dtype=np.uint64)
    n = len(H)
    lut = np.array([bin(i).count("1") for i in range(256)], np.uint8)
    parent = np.arange(n)

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a

    for s in range(0, n, 256):
        x = H[s:s + 256, None] ^ H[None, :]
        d = lut[x.view(np.uint8).reshape(x.shape[0], n, 8)].sum(-1)
        ii, jj = np.nonzero(d <= thr)
        for i, j in zip(ii + s, jj):
            if j > i:
                ra, rb = find(i), find(j)
                if ra != rb:
                    parent[ra] = rb
    return np.array([find(i) for i in range(n)])


def stratified_group_split(df, ratios, seed=0):
    """70/15/15 by disease; whole duplicate groups move together; split by image share within each disease."""
    rng = np.random.RandomState(seed)
    gdis = df.groupby("group").disease.agg(lambda s: s.iloc[0])
    gsize = df.groupby("group").size()
    assign = {}
    for d in sorted(gdis.unique()):
        gids = gdis.index[gdis == d].to_numpy().copy(); rng.shuffle(gids)
        tot = gsize[gids].sum(); acc = 0
        for g in gids:
            pos = (acc + gsize[g] / 2) / tot          # position of the group's midpoint in this disease
            assign[g] = "train" if pos < ratios[0] else ("val" if pos < ratios[0] + ratios[1] else "test")
            acc += gsize[g]
    return df.group.map(assign).to_numpy()


def prepare_data(cfg, n_proc=4):
    """Index, cache, stats and the frozen leak-free split. Safe to call every session (resumes)."""
    from multiprocessing import Pool
    from tqdm.auto import tqdm
    split_csv = wp(cfg, "splits", "split.csv"); split_csv.parent.mkdir(parents=True, exist_ok=True)
    cache = Path(cfg["paths"]["cache"]); (cache / "img").mkdir(parents=True, exist_ok=True); (cache / "msk").mkdir(exist_ok=True)
    fresh = not split_csv.exists()
    if fresh:
        df, missing, img_dir, msk_dir = index_pairs(cfg["paths"]["plantseg_root"])
        log(f"images dir {img_dir} | masks dir {msk_dir} | {len(df)} pairs, {len(missing)} images without a mask")
    else:
        df = pd.read_csv(split_csv)
        log(f"Loaded frozen split {split_csv}: {df.split.value_counts().to_dict()}")
    df["cimg"] = [str(cache / "img" / f"{u:05d}.jpg") for u in df.uid]
    df["cmsk"] = [str(cache / "msk" / f"{u:05d}.png") for u in df.uid]
    args = [(r.image, r.mask, r.cimg, r.cmsk, cfg["cache_long_side"], cfg["img_size"]) for r in df.itertuples()]
    with Pool(n_proc) as pool:
        info = list(tqdm(pool.imap(_cache_one, args, chunksize=16), total=len(args), desc="caching + mask stats"))
    bad = np.array([i is None for i in info])
    if bad.any():
        log(f"WARNING {bad.sum()} unreadable pairs dropped, e.g. {df.image[bad].head(3).tolist()}")
    if fresh:
        keep = ~bad
        df = df[keep].reset_index(drop=True); info = [i for i in info if i is not None]
        for k in ["disease", "lesion_frac", "bg_share", "n_leaves", "n_lesions"]:
            df[k] = [i[k] for i in info]
        comp = np.concatenate([np.array(i["comp_fracs"], np.float32) for i in info])
        q = np.quantile(comp, [0.1, 0.25, 0.5, 0.75, 0.9]) if len(comp) else []
        pd.DataFrame(dict(q=[0.1, 0.25, 0.5, 0.75, 0.9], comp_frac=q)).to_csv(wp(cfg, "splits", "lesion_size_quantiles.csv"), index=False)
        log("lesion component area / image area quantiles (10,25,50,75,90%): " + ", ".join(f"{v:.4%}" for v in q))
        log(f"frozen size bins: small < {cfg['size_bins'][0]:.1%} <= medium <= {cfg['size_bins'][1]:.1%} < large "
            f"-> shares {np.mean(comp < cfg['size_bins'][0]):.1%} / {np.mean((comp >= cfg['size_bins'][0]) & (comp <= cfg['size_bins'][1])):.1%} / {np.mean(comp > cfg['size_bins'][1]):.1%}")
        with Pool(n_proc) as pool:
            hashes = list(tqdm(pool.imap(_phash, df.cimg, chunksize=32), total=len(df), desc="pHash"))
        df["phash"] = [f"{h:016x}" for h in hashes]
        df["group"] = group_near_duplicates(hashes, cfg["phash_thr"])
        gs = df.group.value_counts()
        log(f"near-duplicate groups: {int((gs > 1).sum())} groups hold {int(gs[gs > 1].sum())} images (Hamming <= {cfg['phash_thr']})")
        df["split"] = stratified_group_split(df, cfg["split"], seed=0)
        atomic_to_csv(df.drop(columns=["cimg", "cmsk"]), split_csv, index=False)
        log(f"Frozen split saved: {df.split.value_counts().to_dict()} | diseases per split "
            f"{df.groupby('split').disease.nunique().to_dict()}")
    else:
        df = df[~bad].reset_index(drop=True)
    return df


def get_frames(cfg, df):
    fr = {s: df[df.split == s].reset_index(drop=True) for s in ["train", "val", "test"]}
    if cfg["quick"]:
        n = cfg["quick_n"]
        fr = {s: f.sample(min(k, len(f)), random_state=0).reset_index(drop=True) for (s, f), k in zip(fr.items(), n)}
    for f in fr.values():
        f["key"] = [Path(p).stem for p in f.cimg]
    return fr


def load_split_frames(cfg):
    df = pd.read_csv(wp(cfg, "splits", "split.csv"))
    cache = Path(cfg["paths"]["cache"])
    df["cimg"] = [str(cache / "img" / f"{u:05d}.jpg") for u in df.uid]
    df["cmsk"] = [str(cache / "msk" / f"{u:05d}.png") for u in df.uid]
    df = df[[os.path.exists(p) for p in df.cimg]].reset_index(drop=True)
    return get_frames(cfg, df)

# --------------------------------------------------------------------------------------- dataset
def size_weight_map(mask):
    """Small lesions weigh more: 1/sqrt(area/median area), clipped to [0, 3]; 0 outside lesions."""
    n, lab, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    w = np.zeros(mask.shape, np.float32)
    if n <= 1:
        return w
    areas = stats[1:, cv2.CC_STAT_AREA].astype(np.float32)
    lut = np.zeros(n, np.float32)
    lut[1:] = np.clip(1.0 / np.sqrt(areas / np.median(areas)), 0, 3)
    return lut[lab]


def make_aug(size, train):
    import albumentations as A
    if not train:
        return A.Compose([A.Resize(size, size)])
    v2 = int(A.__version__.split(".")[0]) >= 2
    pad = (A.PadIfNeeded(size, size, border_mode=cv2.BORDER_CONSTANT, fill=0, fill_mask=0) if v2 else
           A.PadIfNeeded(size, size, border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0))
    scales = [int(size * s) for s in np.arange(0.5, 2.01, 0.125)]   # random scale 0.5-2.0 (methodology 4.3)
    return A.Compose([A.LongestMaxSize(max_size=scales), pad, A.RandomCrop(size, size),
                      A.HorizontalFlip(), A.VerticalFlip(p=0.2), A.ColorJitter(0.2, 0.2, 0.2, 0.05, p=0.5)])


def to_tensor(img):
    return torch.from_numpy(((img.astype(np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1).copy())


def read_pair(r):
    img = cv2.cvtColor(cv2.imread(r["cimg"] if "cimg" in r else r["image"]), cv2.COLOR_BGR2RGB)
    m = cv2.imread(r["cmsk"] if "cmsk" in r else r["mask"], cv2.IMREAD_GRAYSCALE)
    if m.shape != img.shape[:2]:
        m = cv2.resize(m, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
    return img, (m > 0).astype(np.uint8)


class LesionSeg(torch.utils.data.Dataset):
    def __init__(self, frame, size, train, clutter=None):
        self.f = frame.reset_index(drop=True); self.aug = make_aug(size, train)
        self.clutter, self.train = clutter, train

    def __len__(self):
        return len(self.f)

    def __getitem__(self, i):
        img, msk = read_pair(self.f.iloc[i])
        o = self.aug(image=img, mask=msk); img, msk = o["image"], o["mask"]
        out = dict(img=to_tensor(img), mask=torch.from_numpy(msk.astype(np.int64)),
                   leaf=torch.from_numpy(leaf_proxy(img)), idx=i)
        if self.train:
            out["size_w"] = torch.from_numpy(size_weight_map(msk))
        if self.clutter is not None:
            c_img, c_msk, occ = self.clutter(img, msk)
            out.update(c_img=to_tensor(c_img), c_mask=torch.from_numpy(c_msk.astype(np.int64)), occ=torch.from_numpy(occ))
        return out


def _worker_init(_):
    info = torch.utils.data.get_worker_info()
    if info is not None and getattr(info.dataset, "clutter", None) is not None:
        info.dataset.clutter.rng.seed(torch.initial_seed() % 2 ** 32)    # different clutter per worker and epoch


def make_loader(frame, cfg, train, bs, clutter=None, shuffle=None, workers=None):
    ds = LesionSeg(frame, cfg["img_size"], train, clutter)
    w = cfg["num_workers"] if workers is None else workers
    return torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=train if shuffle is None else shuffle,
                                       num_workers=w, drop_last=train and len(ds) > bs, worker_init_fn=_worker_init,
                                       pin_memory=torch.cuda.is_available(), persistent_workers=w > 0,
                                       prefetch_factor=4 if w > 0 else None)   # speed: workers stay alive between validations

# --------------------------------------------------------------------------------------- clutter generator
class Clutter:
    """Leaf-like occluders (elliptical patches cut from other training images), shadows, dirt, glare,
    background swap, blur and JPEG artefacts. Returns cluttered image, lesion mask with occluded lesion
    removed, and the occluder mask (1 = pasted pixel, excluded from the CPD KL term).
    `last_altered` holds the share of altered pixels (clutter level per image)."""
    LEVELS = {"light": dict(n_occ=(1, 1), shadow=0.0, dirt=(0, 10), glare=0.0, bg=0.0),
              "medium": dict(n_occ=(1, 3), shadow=0.6, dirt=(0, 30), glare=0.3, bg=0.3),
              "heavy": dict(n_occ=(3, 3), shadow=1.0, dirt=(30, 60), glare=0.5, bg=0.6)}

    def __init__(self, bank, p=0.5, level="medium", max_cover=0.3, seed=None):
        self.bank, self.p, self.cfg, self.max_cover = bank, p, self.LEVELS[level], max_cover
        self.rng = random.Random(seed); self.last_altered = 0.0

    def _ellipse(self, h, w):
        m = np.zeros((h, w), np.uint8); r = self.rng
        cv2.ellipse(m, (r.randint(0, w - 1), r.randint(0, h - 1)), (r.randint(w // 10, w // 4), r.randint(h // 12, h // 5)),
                    r.uniform(0, 180), 0, 360, 1, -1)
        return cv2.GaussianBlur(m.astype(np.float32), (7, 7), 0)

    def _src(self, h, w):
        s = cv2.resize(self.rng.choice(self.bank), (w, h))
        return s[:, ::-1].copy() if self.rng.random() < 0.5 else s

    def __call__(self, img, msk):
        r, c = self.rng, self.cfg
        img, msk = img.copy(), msk.copy(); h, w = msk.shape
        occ = np.zeros((h, w), np.float32); alt = np.zeros((h, w), np.float32)
        self.last_altered = 0.0
        if r.random() > self.p:
            return img, msk, occ
        lesion_px = max(int(msk.sum()), 1)
        if r.random() < c["bg"]:                           # background swap on non-leaf, non-lesion pixels
            leaf = cv2.dilate(leaf_proxy(img), np.ones((7, 7), np.uint8))
            bgm = cv2.GaussianBlur(((leaf == 0) & (msk == 0)).astype(np.float32), (9, 9), 0)
            if 0.02 < bgm.mean() < 0.85:
                img = (img * (1 - bgm[..., None]) + self._src(h, w) * bgm[..., None]).astype(np.uint8)
                alt = np.maximum(alt, (bgm > 0.5).astype(np.float32))
        for _ in range(r.randint(*c["n_occ"])):           # leaf-like occluders, never > 30% of lesion pixels
            src, a = self._src(h, w), self._ellipse(h, w)
            if msk.sum() > 0 and ((a > 0.5) & (msk > 0)).sum() / lesion_px > self.max_cover:
                continue
            img = (img * (1 - a[..., None]) + src * a[..., None]).astype(np.uint8)
            occ = np.maximum(occ, (a > 0.5).astype(np.float32))
        msk[occ > 0] = 0                                    # label follows the occluder
        alt = np.maximum(alt, occ)
        if r.random() < c["shadow"]:
            pts = np.array([[r.randint(0, w), r.randint(0, h)] for _ in range(5)], np.int32)
            s = np.zeros((h, w), np.float32); cv2.fillPoly(s, [pts], 1.0)
            s = cv2.GaussianBlur(s, (31, 31), 0) * r.uniform(0.2, 0.5)
            img = (img * (1 - s[..., None])).astype(np.uint8)
            alt = np.maximum(alt, (s > 0.05).astype(np.float32))
        for _ in range(r.randint(*c["dirt"])):
            g = r.randint(60, 110); ctr = (r.randint(0, w - 1), r.randint(0, h - 1)); rad = r.randint(1, 4)
            cv2.circle(img, ctr, rad, (g, g, g), -1); cv2.circle(alt, ctr, rad, 1.0, -1)
        if r.random() < c["glare"]:
            g = np.zeros((h, w), np.float32)
            cv2.circle(g, (r.randint(0, w - 1), r.randint(0, h - 1)), r.randint(10, 40), 1.0, -1)
            g = cv2.GaussianBlur(g, (41, 41), 0) * 0.6
            img = np.clip(img.astype(np.float32) + g[..., None] * 255, 0, 255).astype(np.uint8)
            alt = np.maximum(alt, (g > 0.05).astype(np.float32))
        if r.random() < 0.3:
            img = cv2.GaussianBlur(img, (3, 3), 0)
        if r.random() < 0.5:
            ok, enc = cv2.imencode(".jpg", img[..., ::-1], [cv2.IMWRITE_JPEG_QUALITY, r.randint(40, 90)])
            img = cv2.imdecode(enc, cv2.IMREAD_COLOR)[..., ::-1].copy()
        self.last_altered = float(alt.mean())
        return img, msk, occ


def make_bank(frame, cfg, n=300, seed=0):
    """Occluder/background source images: TRAINING images only (no test pixels leak into training)."""
    paths = frame.sample(min(n, len(frame)), random_state=seed).cimg
    s = cfg["img_size"]
    return [cv2.resize(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB), (s, s)) for p in paths]


def build_clutter_test(cfg, test, bank):
    """Frozen synthetic clutter test set: three levels, fixed seeds, built once and reused."""
    from tqdm.auto import tqdm
    s = cfg["img_size"]; out = {}
    for level, sd in [("light", 11), ("medium", 22), ("heavy", 33)]:
        d = wp(cfg, "clutter_test", level); csv = d / "index.csv"
        if csv.exists():
            out[level] = pd.read_csv(csv, dtype={"key": str}); continue
        (d / "images").mkdir(parents=True, exist_ok=True); (d / "masks").mkdir(exist_ok=True); (d / "occluders").mkdir(exist_ok=True)
        cl = Clutter(bank, p=1.0, level=level, seed=sd); rows = []
        for _, r in tqdm(test.iterrows(), total=len(test), desc=f"clutter test {level}"):
            img, msk = read_pair(r)
            img = cv2.resize(img, (s, s), interpolation=cv2.INTER_AREA); msk = cv2.resize(msk, (s, s), interpolation=cv2.INTER_NEAREST)
            ci, cm, occ = cl(img, msk)
            ip, mp, op = d / "images" / f"{r.key}.png", d / "masks" / f"{r.key}.png", d / "occluders" / f"{r.key}.png"
            cv2.imwrite(str(ip), cv2.cvtColor(ci, cv2.COLOR_RGB2BGR)); cv2.imwrite(str(mp), cm * 255); cv2.imwrite(str(op), (occ * 255).astype(np.uint8))
            rows.append(dict(image=str(ip), mask=str(mp), key=r.key, disease=r.disease, altered=cl.last_altered,
                             bg_share=r.bg_share, lesion_frac=r.lesion_frac))
        out[level] = pd.DataFrame(rows); out[level].to_csv(csv, index=False)
    return out

# --------------------------------------------------------------------------------------- models
def build(arch, encoder, classes=2, pretrained=True):
    import segmentation_models_pytorch as smp
    return getattr(smp, arch)(encoder_name=encoder, encoder_weights="imagenet" if pretrained else None, classes=classes)


def forward_with_feats(model, x):
    """Logits and the encoder pyramid [input, s2, s4, s8, s16, s32] (indices 3, 4, 5 = strides 8, 16, 32)."""
    feats = model.encoder(x)
    try:
        dec = model.decoder(feats)          # newer smp
    except TypeError:
        dec = model.decoder(*feats)         # older smp
    return model.segmentation_head(dec), feats


class Adapters(nn.Module):
    """1x1 convs from student to teacher channels; training only, dropped at export."""
    def __init__(self, s_ch, t_ch, stages):
        super().__init__(); self.stages = list(stages)
        self.convs = nn.ModuleList([nn.Conv2d(s_ch[i], t_ch[i], 1) for i in self.stages])

    def forward(self, s_feats):
        return [conv(s_feats[i]) for conv, i in zip(self.convs, self.stages)]


def n_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6

# --------------------------------------------------------------------------------------- losses
_DICE = None


def seg_loss(logits, y):
    global _DICE
    if _DICE is None:
        import segmentation_models_pytorch as smp
        _DICE = smp.losses.DiceLoss(mode="multiclass")
    return F.cross_entropy(logits, y) + _DICE(logits, y)


def pixel_kl(t, s, tau):
    """Per-pixel KL(teacher || student) on softened logits, x tau^2. Shape (B, H, W)."""
    return (F.softmax(t / tau, 1) * (F.log_softmax(t / tau, 1) - F.log_softmax(s / tau, 1))).sum(1) * tau ** 2


def logit_kd(t, s, tau):
    return pixel_kl(t, s, tau).mean()


def cwd_loss(t, s, tau):
    """Channel-wise distillation (Shu et al.): KL between each channel's spatial softmax."""
    B, C, H, W = t.shape
    pt = F.softmax(t.reshape(B, C, -1) / tau, -1)
    return (pt * (F.log_softmax(t.reshape(B, C, -1) / tau, -1) - F.log_softmax(s.reshape(B, C, -1) / tau, -1))).sum(-1).mean() * tau ** 2


def attention(f):
    return F.normalize(f.pow(2).mean(1).flatten(1), dim=1)     # Zagoruyko & Komodakis


def resize_to(x, ref):
    return x if x.shape[-2:] == ref.shape[-2:] else F.interpolate(x, size=ref.shape[-2:], mode="bilinear", align_corners=False)


def at_loss(t_feats, s_feats, stages, valid=None):
    loss = 0.0
    for i in stages:
        t_f, s_f = t_feats[i], resize_to(s_feats[i], t_feats[i])
        if valid is not None:
            v = F.interpolate(valid.unsqueeze(1), size=t_f.shape[-2:], mode="nearest")
            t_f, s_f = t_f * v, s_f * v
        loss = loss + (attention(t_f) - attention(s_f)).pow(2).sum(1).mean()
    return loss


def skd_pairwise(t_f, s_f, grid=12):
    """SKD pair-wise term (Liu et al.): match cosine-similarity graphs of pooled feature nodes."""
    t = F.normalize(F.adaptive_avg_pool2d(t_f, grid).flatten(2), dim=1)
    s = F.normalize(F.adaptive_avg_pool2d(s_f, grid).flatten(2), dim=1)
    return (t.transpose(1, 2) @ t - s.transpose(1, 2) @ s).pow(2).mean()


def boundary_band(y, k=5):
    y = y.float().unsqueeze(1)
    band = (F.max_pool2d(y, k, 1, k // 2) + F.max_pool2d(-y, k, 1, k // 2)).clamp(0, 1)   # dilation - erosion
    return F.avg_pool2d(band, 3, 1, 1).squeeze(1)


def bpkd_loss(t, s, y, tau, k=5):
    """BPKD (Liu et al. WACV 2024): KD on edge pixels normalised by edge area + KD on body pixels."""
    e = (boundary_band(y, k) > 0).float(); kl = pixel_kl(t, s, tau)
    return (e * kl).sum() / e.sum().clamp(min=1) + ((1 - e) * kl).sum() / (1 - e).sum().clamp(min=1)


def importance_map(y, size_w, leaf, t_logits, kd):
    """LSD importance W = 1 + a*boundary + b*small-lesion + g*teacher confidence + d*leaf context; mean 1."""
    conf = F.softmax(t_logits, 1).max(1).values
    W = (1 + kd["alpha_boundary"] * boundary_band(y, kd["boundary_k"]) + kd["beta_size"] * size_w
         + kd["gamma_conf"] * conf + kd["delta_leaf"] * leaf * (1 - y.float()))
    return W / W.mean(dim=(1, 2), keepdim=True)


def sard_map(t_logits):
    """SARD-style structure importance (re-implemented from the paper description, no released code used):
    boundary salience of the teacher prediction + local structure density + teacher confidence; mean 1."""
    p = F.softmax(t_logits, 1)[:, 1:2]
    gx = F.pad(p[..., :, 1:] - p[..., :, :-1], (0, 1)); gy = F.pad(p[..., 1:, :] - p[..., :-1, :], (0, 0, 0, 1))
    sal = (gx ** 2 + gy ** 2).sqrt()
    sal = sal / sal.amax(dim=(2, 3), keepdim=True).clamp(min=1e-6)
    dens = F.avg_pool2d(sal, 15, 1, 7); dens = dens / dens.amax(dim=(2, 3), keepdim=True).clamp(min=1e-6)
    conf = F.softmax(t_logits, 1).max(1, keepdim=True).values
    W = (1 + sal + dens + conf).squeeze(1)
    return W / W.mean(dim=(1, 2), keepdim=True)


def chan_norm(f):
    return (f - f.mean(dim=(2, 3), keepdim=True)) / (f.var(dim=(2, 3), keepdim=True, unbiased=False) + 1e-5).sqrt()


def lsd_loss(t_logits, s_logits, t_feats, s_adapted, stages, W, kd):
    """Contribution 1: importance-weighted pixel KL + importance-weighted feature distillation."""
    kl = (W * pixel_kl(t_logits, s_logits, kd["tau"])).sum() / W.sum()
    feat = 0.0
    for s_f, i in zip(s_adapted, stages):
        t_f = t_feats[i]; s_f = resize_to(s_f, t_f)
        Wl = F.interpolate(W.unsqueeze(1), size=t_f.shape[-2:], mode="bilinear", align_corners=False)
        d = (chan_norm(t_f) - chan_norm(s_f)).pow(2).mean(1, keepdim=True)
        feat = feat + (Wl * d).sum() / Wl.sum()
    return kl + kd["mu_feat"] * feat


def cpd_loss(t_logits_clean, s_logits_clutter, t_feats_clean, s_feats_clutter, stages, occ, kd):
    """Contribution 2: clean-view teacher vs cluttered-view student, pasted pixels excluded."""
    valid = 1 - occ
    kl = (valid * pixel_kl(t_logits_clean, s_logits_clutter, kd["tau"])).sum() / valid.sum().clamp(min=1)
    return kd["cpd_kl"] * kl + kd["rho_att"] * at_loss(t_feats_clean, s_feats_clutter, stages, valid)


def loss_unit_tests(cfg):
    kd = cfg["kd"]
    t = torch.randn(2, 2, 32, 32); y = (torch.rand(2, 32, 32) > 0.8).long()
    W = importance_map(y, torch.rand(2, 32, 32), torch.rand(2, 32, 32), t, kd)
    assert abs(W.mean().item() - 1) < 1e-4 and abs(sard_map(t).mean().item() - 1) < 1e-4
    f = [torch.randn(2, 8, 64 // 2 ** i, 64 // 2 ** i) for i in range(6)]
    checks = dict(pixel_kl=pixel_kl(t, t, 4).abs().max(), cwd=cwd_loss(t, t, 4).abs(), bpkd=bpkd_loss(t, t, y, 4).abs(),
                  skd=skd_pairwise(f[5], f[5]).abs(), at=at_loss(f, f, [3, 4, 5]).abs(),
                  lsd=lsd_loss(t, t, f, [f[i] for i in [3, 4, 5]], [3, 4, 5], W, kd).abs(),
                  cpd=cpd_loss(t, t, f, f, [3, 4, 5], torch.zeros(2, 32, 32), kd).abs())
    for k, v in checks.items():
        assert float(v) < 1e-4, (k, float(v))
    s = torch.randn(2, 2, 32, 32)
    for k, v in dict(lsd=lsd_loss(t, s, f, [f[i] * 1.5 + 1 for i in [3, 4, 5]], [3, 4, 5], W, kd),
                     cpd=cpd_loss(t, s, f, [x.flip(-1) for x in f], [3, 4, 5], torch.zeros(2, 32, 32), kd)).items():
        assert v.dim() == 0 and float(v) > 0, k
    return "loss unit tests passed: all terms are scalars, zero when student == teacher, W has mean 1"

# --------------------------------------------------------------------------------------- methods & jobs
METHODS = {
    # name: teacher needed, LSD, CPD, baseline term, adapters, kd overrides
    "A0_student":   dict(teacher=False),
    "B_logitKD":    dict(teacher=True, base="logit"),
    "B_AT":         dict(teacher=True, base="at"),
    "B_CWD":        dict(teacher=True, base="cwd"),
    "B_SKD":        dict(teacher=True, base="skd"),
    "B_BPKD":       dict(teacher=True, base="bpkd"),
    "B_SARD":       dict(teacher=True, base="sard", adapt=True),
    "B_Caug":       dict(teacher=True, base="caug"),     # key control for CPD: teacher sees the SAME cluttered image
    "A1_uniformKD": dict(teacher=True, lsd=True, adapt=True, kd=dict(alpha_boundary=0, beta_size=0, gamma_conf=0, delta_leaf=0)),
    "A2_boundary":  dict(teacher=True, lsd=True, adapt=True, kd=dict(beta_size=0, gamma_conf=0, delta_leaf=0)),
    "A3_size":      dict(teacher=True, lsd=True, adapt=True, kd=dict(gamma_conf=0, delta_leaf=0)),
    "A4_conf":      dict(teacher=True, lsd=True, adapt=True, kd=dict(delta_leaf=0)),
    "A5_LSD":       dict(teacher=True, lsd=True, adapt=True),
    "A6_CPD":       dict(teacher=True, cpd=True),
    "A7_full":      dict(teacher=True, lsd=True, cpd=True, adapt=True),
    # tuning round 2: tuning round 1 showed CWD best on clean images and CPD best under clutter -> combine them
    "X_cwdCPD":     dict(teacher=True, base="cwd", cpd=True),
    "X_cwdCPD_c75": dict(teacher=True, base="cwd", cpd=True, kd=dict(clutter_prob=0.75)),
    "X_cwdLSDCPD":  dict(teacher=True, base="cwd", lsd=True, cpd=True, adapt=True, kd=dict(lambda_lsd=0.5)),
}
SENSITIVITY = {   # all variants of A7_full, MobileNetV3 student, seed 0 (methodology 5.3 "also report")
    "S_tau1": dict(tau=1.0), "S_tau2": dict(tau=2.0),
    "S_lsd0.5": dict(lambda_lsd=0.5), "S_lsd2": dict(lambda_lsd=2.0), "S_lsd5": dict(lambda_lsd=5.0),
    "S_cpd0.5": dict(lambda_cpd=0.5), "S_cpd2": dict(lambda_cpd=2.0), "S_cpd5": dict(lambda_cpd=5.0),
    "S_clut0.25": dict(clutter_prob=0.25), "S_clut0.75": dict(clutter_prob=0.75),
    "S_cpdLogitsOnly": dict(rho_att=0.0), "S_cpdAttOnly": dict(cpd_kl=0.0),
    # tuning pilot (pilot showed the LSD loss ~8x the segmentation loss and CPD ~ plain clutter augmentation)
    "S_feat0.25": dict(mu_feat=0.25), "S_lsd1": dict(lambda_lsd=1.0),
    "S_combo": dict(mu_feat=0.25, lambda_lsd=0.5, rho_att=0.0, clutter_prob=0.75),
}
TUNE_BASE = ["A0_student", "B_AT", "B_CWD", "B_SARD", "B_Caug", "A6_CPD", "A7_full"]
TUNE_VARIANTS = ["S_feat0.25", "S_lsd0.5", "S_cpdLogitsOnly", "S_clut0.75", "S_combo"]
TUNE_VARIANTS2 = ["X_cwdCPD", "X_cwdCPD_c75", "X_cwdLSDCPD"]
for _k, _v in SENSITIVITY.items():
    METHODS[_k] = dict(METHODS["A7_full"], kd=_v)

# Final method set (cfg["method_set"] == "v2"), chosen by the tuning pilot (2 seeds):
#   CWD + LSD(0.5) + CPD matched CWD on clean images (-0.9, within noise) and beat it by +2.8 / +4.0 mIoU under
#   medium / heavy clutter, and C-aug by +1.6 to +1.9 everywhere. Components are ablated on top of CWD.
METHODS_V2 = dict(METHODS)
METHODS_V2.update({
    "A5_LSD":     dict(teacher=True, base="cwd", lsd=True, adapt=True, kd=dict(lambda_lsd=0.5)),   # CWD + LSD
    "A6_CPD":     dict(teacher=True, base="cwd", cpd=True),                                         # CWD + CPD
    "A7_full":    dict(teacher=True, base="cwd", lsd=True, cpd=True, adapt=True, kd=dict(lambda_lsd=0.5)),
    "A8_CPDonly": dict(teacher=True, cpd=True),       # CPD without CWD: like-for-like with the C-aug control
})
for _k, _v in SENSITIVITY.items():
    METHODS_V2[_k] = dict(METHODS_V2["A7_full"], kd={**METHODS_V2["A7_full"]["kd"], **_v})

def method_spec(cfg, method):
    return (METHODS_V2 if cfg.get("method_set") == "v2" else METHODS).get(method, {})

FIRST_KD = ["B_logitKD", "B_AT", "B_CWD", "B_BPKD", "B_SARD", "B_Caug", "A5_LSD", "A6_CPD", "A7_full"]
ABLATIONS = ["A1_uniformKD", "A2_boundary", "A3_size", "A4_conf"]
BASELINES = ["B_logitKD", "B_AT", "B_CWD", "B_BPKD", "B_SARD", "B_Caug"]


# --------------------------------------------------------------------------------------- teacher selection
def teacher_run(cfg, key):
    return f"teacher-{key}__teacher__s0"

def chosen_teacher(cfg):
    """The teacher every distillation run uses: cfg['teacher_choice'] if fixed, else ('auto') the trained
    candidate with the highest validation mIoU. None until a candidate has finished."""
    if cfg["teacher_choice"] != "auto":
        return cfg["teacher_choice"] if run_done(cfg, teacher_run(cfg, cfg["teacher_choice"])) else None
    if any(not run_done(cfg, teacher_run(cfg, k)) for k in cfg["teachers_to_train"]):
        return None                                   # wait until every candidate is trained, then choose
    sc = {k: json.load(open(wp(cfg, "runs", teacher_run(cfg, k), "done.json")))["best_val_miou"] for k in cfg["teachers_to_train"]}
    return max(sc, key=sc.get)

def teacher_spec(cfg):
    k = chosen_teacher(cfg)
    return None if k is None else dict(key=k, run=teacher_run(cfg, k), **cfg["teachers"][k])


def run_done(cfg, name):
    return wp(cfg, "runs", name, "done.json").exists()


def eval_done(cfg, name):
    return wp(cfg, "results", "evaldone", name).exists()


def val_scores(cfg):
    rows = []
    for p in wp(cfg, "runs").glob("*/done.json"):
        rows.append(json.load(open(p)))
    return pd.DataFrame(rows)


def best_baseline(cfg, student_short):
    v = val_scores(cfg)
    if v.empty:
        return None
    v = v[(v.model == student_short) & v.method.isin(BASELINES)]
    if v.empty:
        return None
    return v.groupby("method").best_val_miou.mean().idxmax()


def phases(cfg):
    """Ordered phases of the experiment grid. Each phase: (title, gate checked before it, job list).
    Trimmed grid: primary student gets every method; secondary student only A0, best baseline, A7.
    The 'full' gate is checked after seed 0, so a failed idea stops before the other seeds are spent."""
    S, S2 = cfg["students"], cfg.get("secondary_students", []); seeds = cfg["seeds"]; q = cfg["quick"]
    final_seeds = cfg["final_seeds"][:1] if q else cfg["final_seeds"]
    if q:
        seeds = seeds[:1]                       # sanity / pilot: one seed
    J = lambda st, m, sd: dict(name=f"{short(st)}__{m}__s{sd}", kind="student", encoder=st, arch="Unet", method=m, seed=sd)
    p1 = [dict(name=teacher_run(cfg, k), kind="teacher", encoder=cfg["teachers"][k]["encoder"], arch=cfg["teachers"][k]["arch"],
               method="teacher", seed=0, transformer=cfg["teachers"][k].get("transformer", False)) for k in cfg["teachers_to_train"]]
    p1 += [J(st, "A0_student", sd) for sd in seeds for st in S]
    p1 += [dict(name=f"{d['name']}__direct__s{sd}", kind="direct", encoder=d["encoder"], arch=d["arch"], method="direct",
                seed=sd, transformer=d["transformer"]) for sd in seeds for d in cfg["direct"]]
    sec = lambda sds: [J(st, m, sd) for sd in sds for st in S2 for m in cfg["secondary_methods"]]
    sens = [m for m in cfg["sensitivity"] if m in SENSITIVITY]
    first_kd = cfg.get("first_kd") or FIRST_KD                 # seed 0
    seed_kd = cfg.get("seed_kd") or first_kd                   # seeds 1, 2
    ablations = ABLATIONS if cfg.get("ablations") is None else cfg["ablations"]
    out = [("P1 teacher(s) + student alone + direct small models", None, p1),
           ("P2 baselines + LSD/CPD/full, primary student, seed 0", "teacher", [J(st, m, seeds[0]) for st in S for m in first_kd]),
           ("P3 other seeds + second student (A0, best baseline, A7)", "full",
            [J(st, m, sd) for sd in seeds[1:] for st in S for m in seed_kd] + sec(seeds)),
           ("P4 ablations A1-A4, primary student", None, [J(st, m, sd) for sd in seeds for st in S for m in ablations]),
           ("P5 sensitivity (A7 variants, primary student, seed 0)", None, [J(S[0], m, seeds[0]) for m in (sens[:2] if q else sens)])]
    if cfg.get("tune"):                         # tuning pilot: reuses the pilot's teachers and seed-0 runs
        st = S[0]
        return [out[0],
                ("T1 baselines + A7, second seed", None, [J(st, m, 1) for m in TUNE_BASE]),
                ("T2 tuned A7 variants, seeds 0 and 1", None, [J(st, m, sd) for sd in (0, 1) for m in TUNE_VARIANTS]),
                ("T3 CWD + CPD combinations, seeds 0 and 1", None, [J(st, m, sd) for sd in (0, 1) for m in TUNE_VARIANTS2])]
    fin = [J(st, m, sd) for sd in final_seeds for st in S + S2 for m in ["A0_student", "best_baseline", "A7_full"]]
    if fin:
        out.append(("P6 extra seeds for the final table", None, fin))
    out = out[:cfg.get("max_phase", 99)]
    if cfg.get("max_phase", 99) < 3:            # pilot: still check A7 vs the best baseline after P2
        out.append(("P2 check: A7 vs best baseline", "full", []))
    return out


def resolve(cfg, job):
    if job["method"] == "best_baseline":
        # the best baseline is chosen on the PRIMARY student's validation scores (seed-averaged)
        b = best_baseline(cfg, short(job["encoder"])) or best_baseline(cfg, short(cfg["students"][0])) or "B_CWD"
        job = dict(job, method=b, name=f"{short(job['encoder'])}__{b}__s{job['seed']}")
    return job


def gate(cfg, which):
    """Week-3 gate: teacher beats every student-alone model by >= 5 mIoU (val).
    Week-7 gate: full model beats the best baseline by >= 1.5 mIoU (val, mean over seeds), per student."""
    v = val_scores(cfg); g = cfg["gates"]
    if which == "teacher":
        spec = teacher_spec(cfg)
        t = v[v.name == spec["run"]].best_val_miou.max() if spec else float("nan")
        a0 = v[v.method == "A0_student"].groupby("model").best_val_miou.mean()
        dirm = v[v.method == "direct"].groupby("model").best_val_miou.mean()
        best_small = max(a0.max() if len(a0) else 0, dirm.max() if len(dirm) else 0)
        ok = t - best_small >= g["teacher_margin"]
        return ok, f"teacher {spec['key'] if spec else '?'} val mIoU {t:.4f} vs best small model {best_small:.4f} (margin {t - best_small:+.4f}, need {g['teacher_margin']:+.3f})"
    msgs, oks = [], []
    for st in cfg["students"]:
        s = short(st); vv = v[v.model == s]
        full = vv[vv.method == "A7_full"].best_val_miou.mean()
        base = vv[vv.method.isin(BASELINES)].groupby("method").best_val_miou.mean()
        bb = base.idxmax(); m = full - base.max()
        oks.append(m >= g["full_margin"])
        msgs.append(f"{s}: A7_full {full:.4f} vs best baseline {bb} {base.max():.4f} (margin {m:+.4f}, need {g['full_margin']:+.3f})")
    return all(oks), " | ".join(msgs)


def claim(cfg, name, gpu):
    d = wp(cfg, "claims"); d.mkdir(parents=True, exist_ok=True)
    try:
        fd = os.open(str(d / f"{name}.claim"), os.O_CREAT | os.O_EXCL | os.O_WRONLY)
        os.write(fd, f"pid={os.getpid()} gpu={gpu} t={time.time()}".encode()); os.close(fd)
        return True
    except FileExistsError:
        return False


def release(cfg, name):
    try:
        os.remove(wp(cfg, "claims", f"{name}.claim"))
    except FileNotFoundError:
        pass


def next_job(cfg, gpu):
    """Returns ('job', job) | ('wait', msg) | ('finished', msg) | ('stop', msg)."""
    for title, g, jobs in phases(cfg):
        if g is not None:
            stop_file = wp(cfg, f"GATE_{g}.txt")
            if not stop_file.exists():
                ok, msg = gate(cfg, g)
                atomic_write_text(stop_file, ("PASS " if ok else "FAIL ") + msg + "\n")
                log(f"GATE {g}: {'PASS' if ok else 'FAIL'} - {msg}")
            txt = stop_file.read_text()
            if txt.startswith("FAIL") and cfg["gates"]["stop_on_fail"]:
                return "stop", f"gate '{g}' failed: {txt.strip()}. Methodology 6.2: fix the teacher/idea before continuing. " \
                               f"To continue anyway set CFG['gates']['stop_on_fail']=False and delete {stop_file.name}."
        jobs = [resolve(cfg, j) for j in jobs]
        pending = [j for j in jobs if not (run_done(cfg, j["name"]) and eval_done(cfg, j["name"]))]
        if not pending:
            continue
        for j in pending:
            if j["kind"] == "student" and method_spec(cfg, j["method"]).get("teacher") and teacher_spec(cfg) is None:
                continue          # cannot start before the teacher exists
            if claim(cfg, j["name"], gpu):
                return "job", j
        if any(not run_done(cfg, j["name"]) for j in jobs):
            return "wait", f"{title}: remaining jobs are running on another GPU"
        return "wait", f"{title}: evaluation still running elsewhere"
    return "finished", "all phases complete"

# --------------------------------------------------------------------------------------- training
_TEACHER = {}


def load_teacher(cfg, dev):
    if "m" not in _TEACHER:
        spec = teacher_spec(cfg)
        t = build(spec["arch"], spec["encoder"], pretrained=False).to(dev)
        t.load_state_dict(torch.load(wp(cfg, "runs", spec["run"], "best.pt"), map_location=dev)); t.eval()
        for p in t.parameters():
            p.requires_grad_(False)
        _TEACHER["m"] = t
    return _TEACHER["m"]


@torch.no_grad()
def val_miou(model, dl, dev, amp):
    model.eval(); inter = torch.zeros(2, device=dev); union = torch.zeros(2, device=dev)
    for b in dl:
        with torch.autocast("cuda", enabled=amp):
            p = model(b["img"].to(dev, non_blocking=True)).argmax(1)
        y = b["mask"].to(dev)
        for c in (0, 1):
            inter[c] += ((p == c) & (y == c)).sum(); union[c] += ((p == c) | (y == c)).sum()
    iou = (inter / union.clamp(min=1)).cpu().numpy()
    return dict(miou=float(iou.mean()), iou_lesion=float(iou[1]))


def train_run(cfg, job, frames, bank, dev, deadline):
    name = job["name"]; rd = wp(cfg, "runs", name); rd.mkdir(parents=True, exist_ok=True)
    if (rd / "done.json").exists():
        return "done"
    kind, method, seed = job["kind"], job["method"], job["seed"]
    M = method_spec(cfg, method) if kind == "student" else {}
    kd = {**cfg["kd"], **M.get("kd", {})}; stages = kd["feat_stages"]
    seed_all(seed)
    amp = dev.type == "cuda" and torch.cuda.get_device_capability(dev)[0] >= 7
    if kind == "teacher":
        epochs, bs = cfg["teacher_epochs"], cfg["teacher_bs"]
        lr = cfg["lr_transformer"] if job.get("transformer") else cfg["lr_teacher"]
    else:
        epochs, bs = cfg["student_epochs"], cfg["student_bs"]
        lr = cfg["lr_transformer"] if job.get("transformer") else cfg["lr_student"]
    if cfg["quick"]:
        epochs = cfg["quick_epochs"]
    teacher = load_teacher(cfg, dev) if M.get("teacher") else None
    need_clutter = bool(M.get("cpd")) or M.get("base") == "caug"
    clutter = Clutter(bank, p=kd["clutter_prob"], level="medium", seed=seed) if need_clutter else None
    workers = cfg["num_workers"]
    tr = make_loader(frames["train"], cfg, True, bs, clutter=clutter, workers=workers)
    va = make_loader(frames["val"], cfg, False, cfg["eval_bs"])
    model = build(job["arch"], job["encoder"], pretrained=cfg["pretrained"]).to(dev)
    adapt = Adapters(model.encoder.out_channels, teacher.encoder.out_channels, stages).to(dev) if M.get("adapt") else None
    params = list(model.parameters()) + (list(adapt.parameters()) if adapt else [])
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=cfg["weight_decay"])
    total = epochs * len(tr); warm = max(1, min(cfg["warmup_iters"], total // 10))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda it: min(1.0, (it + 1) / warm) * max(0.0, 1 - it / total) ** 0.9)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    start, best, hist, bad = 0, -1.0, [], 0
    ck = rd / "last.pt"
    if ck.exists():                                                 # resume after a disconnect / new session
        s = torch.load(ck, map_location=dev, weights_only=False)
        model.load_state_dict(s["model"]); opt.load_state_dict(s["opt"]); sched.load_state_dict(s["sched"])
        scaler.load_state_dict(s["scaler"])
        if adapt is not None and s.get("adapt"):
            adapt.load_state_dict(s["adapt"])
        start, best, hist, bad = s["epoch"] + 1, s["best"], s["log"], s.get("bad", 0)
        log(f"[{name}] resumed at epoch {start}")
    t_run = time.time()
    for ep in range(start, epochs):
        est = np.mean([h["sec"] for h in hist[-3:]]) if hist else 0
        if deadline and time.time() + 1.15 * est + 600 > deadline:
            log(f"[{name}] paused before epoch {ep} (session time budget); resumes next session")
            return "paused"
        st = sys_stats()
        if st["ram_pct"] >= cfg.get("ram_high", 88) and workers > 0:     # slow down: fewer data-loading processes
            workers = max(0, workers // 2); del tr
            import gc; gc.collect()
            tr = make_loader(frames["train"], cfg, True, bs, clutter=clutter, workers=workers)
            log(f"[{name}] RAM {st['ram_pct']}% (free {st['ram_free_gb']} GB): data workers reduced to {workers}")
        cool_s = temp_guard(cfg, name)
        model.train(); t0 = time.time(); parts = collections.defaultdict(lambda: torch.zeros((), device=dev)); nb = 0; paused = 0
        if adapt is not None:
            adapt.train()
        for b in tr:
            x, y = b["img"].to(dev, non_blocking=True), b["mask"].to(dev, non_blocking=True)
            with torch.autocast("cuda", enabled=amp):
                if teacher is not None:
                    with torch.no_grad():
                        t_out, t_feats = forward_with_feats(teacher, x)
                s_out, s_feats = forward_with_feats(model, x)
            s_out = s_out.float(); loss = seg_loss(s_out, y); parts["seg"] += loss.detach()
            if teacher is not None:
                t_out = t_out.float(); t_feats = [f.float() for f in t_feats]; s_feats = [f.float() for f in s_feats]
                base, l = M.get("base"), None
                if base == "logit":
                    l = logit_kd(t_out, s_out, kd["tau"])
                elif base == "at":
                    l = at_loss(t_feats, s_feats, stages)
                elif base == "cwd":
                    l = cwd_loss(t_out, s_out, kd["tau"])
                elif base == "skd":
                    l = logit_kd(t_out, s_out, kd["tau"]) + kd["skd_w"] * skd_pairwise(t_feats[5], s_feats[5])
                elif base == "bpkd":
                    l = bpkd_loss(t_out, s_out, y, kd["tau"], kd["boundary_k"])
                elif base == "sard":
                    l = lsd_loss(t_out, s_out, t_feats, adapt(s_feats), stages, sard_map(t_out), kd)
                if l is not None:
                    loss = loss + l; parts[base] += l.detach()
                if M.get("lsd"):
                    W = importance_map(y, b["size_w"].to(dev), b["leaf"].to(dev), t_out, kd)
                    l = lsd_loss(t_out, s_out, t_feats, adapt(s_feats), stages, W, kd)
                    loss = loss + kd["lambda_lsd"] * l; parts["lsd"] += l.detach()
            if need_clutter:
                xc, yc, occ = b["c_img"].to(dev), b["c_mask"].to(dev), b["occ"].to(dev)
                with torch.autocast("cuda", enabled=amp):
                    sc_out, sc_feats = forward_with_feats(model, xc)
                    if M.get("base") == "caug":
                        with torch.no_grad():
                            tc_out, _ = forward_with_feats(teacher, xc)
                sc_out = sc_out.float(); sc_feats = [f.float() for f in sc_feats]
                lc = seg_loss(sc_out, yc); loss = loss + lc; parts["seg_clutter"] += lc.detach()
                if M.get("base") == "caug":
                    l = logit_kd(tc_out.float(), sc_out, kd["tau"])
                else:
                    l = cpd_loss(t_out, sc_out, t_feats, sc_feats, stages, occ, kd)
                loss = loss + kd["lambda_cpd"] * l; parts["clutter_kd"] += l.detach()
            if nb % 25 == 0 and not torch.isfinite(loss):    # checked every 25 batches: each check makes the CPU wait for the GPU
                raise FloatingPointError(f"[{name}] non-finite loss at epoch {ep}: { {k: float(v) for k, v in parts.items()} }")
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); nb += 1
            if nb % 50 == 0 and ram_guard(cfg, name):
                paused += 1
            if nb % 25 == 0:
                cool_s += temp_guard(cfg, name)
        do_val = (ep + 1) % cfg.get("val_every", 1) == 0 or ep == epochs - 1
        v = val_miou(model, va, dev, amp) if do_val else dict(miou=None, iou_lesion=None)
        rec = dict(epoch=ep, val_miou=v["miou"], val_iou_lesion=v["iou_lesion"], sec=round(time.time() - t0, 1),
                   lr=opt.param_groups[0]["lr"], workers=workers, ram_pauses=paused, cooldown_s=round(cool_s),
                   gpu_temp=gpu_temp(), **sys_stats(),
                   **{k: round(float(x) / max(nb, 1), 5) for k, x in parts.items()})
        hist.append(rec); log(f"[{name}] {rec}")
        if do_val:
            if v["miou"] > best:
                best, bad = v["miou"], 0; torch.save(model.state_dict(), rd / "best.pt.tmp"); os.replace(rd / "best.pt.tmp", rd / "best.pt")
            else:
                bad += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(),
                        epoch=ep, best=best, bad=bad, log=hist, adapt=adapt.state_dict() if adapt is not None else None), str(ck) + ".tmp")
        os.replace(str(ck) + ".tmp", ck)                # atomic: a power cut mid-save never leaves a broken checkpoint
        pd.DataFrame(hist).to_csv(rd / "log.csv", index=False)
        if cfg.get("patience_evals") and bad >= cfg["patience_evals"]:
            log(f"[{name}] early stop at epoch {ep}: no val improvement in {bad} checks"); break
    model_name, _, _ = parse_run(name)
    res = dict(name=name, model=model_name, kind=kind, arch=job["arch"], encoder=job["encoder"], method=method, seed=seed,
               best_val_miou=best, params_M=round(n_params(model), 3), epochs=epochs, epochs_run=len(hist),
               train_hours=round(sum(h["sec"] for h in hist) / 3600, 3), quick=cfg["quick"], kd=kd if teacher is not None else None)
    atomic_write_text(rd / "done.json", json.dumps(res, indent=1))
    ck.unlink(missing_ok=True)
    return "done"

# --------------------------------------------------------------------------------------- evaluation
def lesion_match(p, y, bins, thr=0.5):
    """Lesion-level TP/FP/FN (a GT lesion is found if one predicted component has IoU >= thr with it),
    plus TP/FN per lesion size bin (component area / image area)."""
    ny, ly = cv2.connectedComponents(y); npr, lp = cv2.connectedComponents(p)
    out = dict(tp=0, fp=0, fn=0, tp_small=0, fn_small=0, tp_medium=0, fn_medium=0, tp_large=0, fn_large=0)
    if ny <= 1:
        out["fp"] = npr - 1; return out
    pa = np.bincount(lp.ravel(), minlength=npr); ga = np.bincount(ly.ravel(), minlength=ny)
    pair = np.bincount((ly.astype(np.int64) * npr + lp).ravel(), minlength=ny * npr).reshape(ny, npr)
    matched = set()
    for g in range(1, ny):
        inter = pair[g, 1:]; iou = inter / (ga[g] + pa[1:] - inter + 1e-9)
        hit = iou.size > 0 and iou.max() >= thr
        if hit:
            matched.add(int(iou.argmax()) + 1)
        frac = ga[g] / y.size
        b = "small" if frac < bins[0] else ("medium" if frac <= bins[1] else "large")
        out["tp"] += hit; out["fn"] += (not hit); out[f"tp_{b}"] += hit; out[f"fn_{b}"] += (not hit)
    out["fp"] = (npr - 1) - len(matched)
    return out


def boundary_f1(p, y, tol=2):
    k = np.ones((3, 3), np.uint8); kt = np.ones((2 * tol + 1, 2 * tol + 1), np.uint8)
    bp, by = p - cv2.erode(p, k), y - cv2.erode(y, k)
    if bp.sum() == 0 and by.sum() == 0:
        return 1.0
    prec = (bp & cv2.dilate(by, kt)).sum() / max(bp.sum(), 1); rec = (by & cv2.dilate(bp, kt)).sum() / max(by.sum(), 1)
    return float(2 * prec * rec / max(prec + rec, 1e-8))


def image_metrics(p, y, leaf, bins):
    i1 = int(((p == 1) & (y == 1)).sum()); u1 = int(((p == 1) | (y == 1)).sum())
    i0 = int(((p == 0) & (y == 0)).sum()); u0 = int(((p == 0) | (y == 0)).sum())
    r = dict(inter1=i1, union1=u1, inter0=i0, union0=u0, iou=i1 / u1 if u1 else 1.0,
             dice=2 * i1 / max(int(p.sum()) + int(y.sum()), 1) if (p.sum() + y.sum()) else 1.0,
             bf1=boundary_f1(p, y), sev_err=abs(int(p.sum()) - int(y.sum())) / max(float(leaf.sum()), 1.0) * 100,
             lesion_frac=float(y.mean()), pred_frac=float(p.mean()))
    r.update(lesion_match(p, y, bins))
    return r


@torch.no_grad()
def eval_frame(model, frame, cfg, dev, workers=None):
    model.eval(); amp = dev.type == "cuda" and torch.cuda.get_device_capability(dev)[0] >= 7
    dl = make_loader(frame, cfg, False, cfg["eval_bs"], shuffle=False, workers=workers)
    rows = []; pool = ThreadPoolExecutor(4)
    for b in dl:
        with torch.autocast("cuda", enabled=amp):
            pred = model(b["img"].to(dev)).argmax(1).cpu().numpy().astype(np.uint8)
        ys = b["mask"].numpy().astype(np.uint8); leaves = b["leaf"].numpy(); idx = b["idx"].numpy()
        res = list(pool.map(lambda k: image_metrics(pred[k], ys[k], leaves[k], cfg["size_bins"]), range(len(pred))))
        for k, r in zip(idx, res):
            r["idx"] = int(k); rows.append(r)
    pool.shutdown()
    if not rows:
        raise ValueError(f"empty evaluation frame ({len(frame)} rows)")
    out = pd.DataFrame(rows).sort_values("idx").reset_index(drop=True)
    for c in ["key", "disease", "bg_share", "n_leaves", "altered"]:
        if c in frame.columns:
            out[c] = frame[c].to_numpy()[out.idx]
    return out


def test_sets(cfg, frames):
    sets = {"plantseg_test": frames["test"]}
    for lv in ["light", "medium", "heavy"]:
        p = wp(cfg, "clutter_test", lv, "index.csv")
        if p.exists():
            f = pd.read_csv(p, dtype={"key": str})
            sets[f"clutter_{lv}"] = f[f.key.isin(set(frames["test"].key))].reset_index(drop=True)
    pd_dir = Path(cfg["paths"].get("plantdoc_seg") or wp(cfg, "plantdoc_seg"))
    if (pd_dir / "masks").exists():
        masks = {p.stem: p for p in (pd_dir / "masks").glob("*.png")}
        rows = [dict(image=str(p), mask=str(masks[p.stem]), key=p.stem, disease=-1)
                for p in sorted((pd_dir / "images").glob("*")) if p.stem in masks]
        if rows:
            sets["plantdoc_seg"] = pd.DataFrame(rows)
    return sets


def load_model(cfg, name, dev):
    info = json.load(open(wp(cfg, "runs", name, "done.json")))
    m = build(info["arch"], info["encoder"], pretrained=False).to(dev)
    m.load_state_dict(torch.load(wp(cfg, "runs", name, "best.pt"), map_location=dev)); m.eval()
    return m, info


def eval_run(cfg, name, frames, dev, workers=None):
    out_dir = wp(cfg, "results", "per_image"); out_dir.mkdir(parents=True, exist_ok=True)
    m, info = load_model(cfg, name, dev)
    for tname, fr in test_sets(cfg, frames).items():
        p = out_dir / f"{name}__{tname}.csv"
        if p.exists():                                    # keep only complete files (an old half-written file is redone)
            try:
                ok = len(pd.read_csv(p)) == len(fr)
            except Exception:
                ok = False
            if not ok:
                log(f"[{name}] incomplete result file {p.name}: evaluating again"); p.unlink()
        if not p.exists():
            atomic_to_csv(eval_frame(m, fr, cfg, dev, workers=workers), p, index=False)
    (wp(cfg, "results", "evaldone")).mkdir(parents=True, exist_ok=True)
    atomic_write_text(wp(cfg, "results", "evaldone", name), str(time.time()))
    del m
    if dev.type == "cuda":
        torch.cuda.empty_cache()

# --------------------------------------------------------------------------------------- worker
def worker(work, gpu):
    cfg = load_cfg(work)
    if keep_awake():
        log("Windows sleep and screen-off blocked while training runs")
    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    torch.backends.cudnn.benchmark = True
    frames = load_split_frames(cfg); bank = make_bank(frames["train"], cfg)
    log(f"worker gpu={gpu} device={torch.cuda.get_device_name(0) if dev.type == 'cuda' else 'cpu'} | "
        f"train {len(frames['train'])} val {len(frames['val'])} test {len(frames['test'])} | quick={cfg['quick']}")
    deadline = cfg.get("deadline")
    while True:
        if deadline and time.time() > deadline - 900:
            log("session time budget reached; exiting (state saved, rerun the notebook to continue)"); return
        status, x = next_job(cfg, gpu)
        if status == "job":
            j = x
            try:
                log(f"START {j['name']}")
                st = train_run(cfg, j, frames, bank, dev, deadline)
                if st == "done":
                    if not eval_done(cfg, j["name"]):
                        eval_run(cfg, j["name"], frames, dev)
                    log(f"DONE {j['name']}")
                    (wp(cfg, "runs", j["name"]) / "error.txt").unlink(missing_ok=True)   # an old crash of this run is now fixed
                else:
                    release(cfg, j["name"]); return
            except Exception:
                log(f"ERROR in {j['name']}:\n{traceback.format_exc()}")
                wp(cfg, "runs", j["name"]).mkdir(parents=True, exist_ok=True)
                wp(cfg, "runs", j["name"], "error.txt").write_text(traceback.format_exc())
                release(cfg, j["name"]); raise
            release(cfg, j["name"])
        elif status == "wait":
            log(f"waiting: {x}"); time.sleep(60)
        else:
            log(f"{status.upper()}: {x}"); return


if __name__ == "__main__":
    if len(sys.argv) >= 4 and sys.argv[1] == "worker":
        worker(sys.argv[2], int(sys.argv[3]))

In [ ]:
# ===== 1. SETUP + CONFIG =====
import os, sys, json, time, shutil, subprocess, importlib
from pathlib import Path
SESSION_START = time.time()
ON_KAGGLE = os.path.exists("/kaggle/input")
IN_COLAB = "google.colab" in sys.modules
ON_WINDOWS = os.name == "nt"

# ---------------- choose what to run ----------------
RUN_MODE = "full"     # 1) "sanity": 64 images, 2 epochs, ~30 min - checks paths/packages, nothing else
                        # 2) "pilot" : 1,600/400/400 images, 256 px, 20 epochs, MobileNetV3 only, seed 0, BOTH teachers
                        #              -> answers overnight: which teacher, and does A7 beat the baselines?
                        # 2b) "tune" : overnight tuning pilot on the pilot data: 2 seeds of A0, CWD, SARD, C-aug, A7
                        #              + 5 lighter-loss A7 variants (reuses the pilot's teachers and runs, ~7 h)
                        # 3) "full"  : the paper runs (320 px, 40 epochs + early stopping, trimmed grid, 3 seeds)
FULL_TEACHER = "segformer_b4"   # for "full": set to the pilot's winner, "segformer_b4" or "resnet50" (only that one is trained)
TIME_BUDGET_H = 11.0 if os.path.exists("/kaggle/input") else 1000.0   # Kaggle stops a session at 12 h; lab PC: no limit
QUICK = RUN_MODE != "full"

def pip(*pk):
    """pip install that can never replace the installed PyTorch (e.g. a CUDA build by a CPU-only one)."""
    from importlib.metadata import version, PackageNotFoundError
    pins = []
    for p in ["torch", "torchvision"]:
        try: pins.append(f"{p}=={version(p)}")
        except PackageNotFoundError: pass
    cons = Path("pip_constraints.txt"); cons.write_text("\n".join(pins) + "\n")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", str(cons), *pk], check=False)
# install only what is missing in this environment (PyTorch itself is never touched: see pip() above)
REQ = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "scipy": "scipy", "tqdm": "tqdm",
       "PIL": "pillow", "skimage": "scikit-image", "imagehash": "imagehash", "thop": "thop", "psutil": "psutil",
       "timm": "timm", "onnx": "onnx", "onnxscript": "onnxscript", "IPython": "ipython"}
missing = []
for mod, pkg in REQ.items():
    try: importlib.import_module(mod)
    except Exception: missing.append(pkg)
try:
    import cv2
except ImportError:
    missing.append("opencv-python-headless")       # only if no OpenCV at all (two OpenCV builds break each other)
try:
    import segmentation_models_pytorch as _smp, albumentations as _al
    assert _smp.__version__ == "0.5.0" and _al.__version__.startswith("2.0")
except Exception:
    missing += ["segmentation-models-pytorch==0.5.0", "albumentations==2.0.8"]
if missing:
    print("installing:", missing); pip(*missing); importlib.invalidate_caches()

if "p1lib" in sys.modules:
    importlib.reload(sys.modules["p1lib"])
import p1lib as L
import numpy as np, pandas as pd, torch

if ON_KAGGLE:
    BASE_IN = Path("/kaggle/input"); WORK = Path("/kaggle/working/paper1_runs"); CACHE = Path("/tmp/p1cache")
elif IN_COLAB:
    from google.colab import drive
    import zipfile
    drive.mount("/content/drive")
    ZIP_DIR = Path("/content/drive/MyDrive/MSc/dataset")            # where plantsegv3.zip and PlantDoc-Dataset-master.zip are
    BASE_IN = Path("/content/data")
    for name in ["plantsegv3", "PlantDoc-Dataset-master"]:
        if not (BASE_IN / name).exists() and (ZIP_DIR / f"{name}.zip").exists():
            print("unzipping", name); zipfile.ZipFile(ZIP_DIR / f"{name}.zip").extractall(BASE_IN / name)
    WORK = Path("/content/drive/MyDrive/PlantDisease/paper1_runs_v2"); CACHE = Path("/content/p1cache")
elif ON_WINDOWS:
    BASE_IN = Path(os.environ.get("P1_ROOT", ".")) / r"Dataset"; WORK = Path(os.environ.get("P1_ROOT", ".")) / r"paper1_runs_v2"; CACHE = WORK / "cache"
else:
    BASE_IN = Path("data"); WORK = Path("paper1_runs_v2"); CACHE = Path("p1cache")
if RUN_MODE != "full":
    WORK = WORK.with_name(WORK.name + "_" + ("pilot" if RUN_MODE == "tune" else RUN_MODE))   # tune continues in the pilot folder   # sanity / pilot runs never mix with the paper runs
# optional overrides (used for automated testing): P1_OVERRIDES='{"paths": {...}, "cfg": {...}}'
OV = json.loads(os.environ.get("P1_OVERRIDES", "{}"))
BASE_IN = Path(OV.get("paths", {}).get("base_in", BASE_IN)); WORK = Path(OV.get("paths", {}).get("work", WORK))
CACHE = Path(OV.get("paths", {}).get("cache", CACHE))

# resume: copy the previous session's output back (Kaggle: add the previous notebook version as an input)
if not (WORK / "config.json").exists():
    prev = sorted(p.parent for p in BASE_IN.rglob("paper1_runs*/config.json")
                  if p.parent.name == WORK.name and p.parent.resolve() != WORK.resolve())
    if prev:
        print("Resuming from", prev[-1]); shutil.copytree(prev[-1], WORK, dirs_exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

_need = [("images", "image", "imgs"), ("annotations", "annotation", "masks", "mask", "labels")]
_ps = sorted(p for p in BASE_IN.iterdir() if p.is_dir() and "plantseg" in p.name.lower()) if BASE_IN.exists() else []
PLANTSEG_ROOT = next((r for r in (L.find_dataset_root(p, _need) for p in _ps) if r), None) or L.find_dataset_root(BASE_IN, _need)
cands = [Path(r) for r, d, _ in os.walk(BASE_IN) if {"train", "test"} <= {x.lower() for x in d}
         and not any(x.lower() in ("images", "annotations") for x in d) and "images" not in Path(r).parts[-1:]]
cands = [c for c in cands if PLANTSEG_ROOT is None or PLANTSEG_ROOT not in c.parents]
PLANTDOC_ROOT = next((c for c in cands if "plantdoc" in str(c).lower()), cands[0] if cands else None)
PD_SEG = next((Path(r) for r, d, _ in os.walk(BASE_IN) if Path(r).name.lower() == "plantdoc_seg" and "masks" in d), None)

CFG = json.loads(json.dumps(L.DEFAULT_CFG))
DEFAULT_GATES = dict(L.DEFAULT_CFG["gates"])
MODES = {
    "sanity": dict(quick=True, quick_n=[64, 32, 32], quick_epochs=2, img_size=256, cache_long_side=512,
                   gates=dict(DEFAULT_GATES, stop_on_fail=False)),   # 2-epoch models are meaningless: never stop on a gate
    "pilot": dict(quick=True, quick_n=[1600, 400, 400], quick_epochs=20, img_size=256, cache_long_side=512,
                  secondary_students=[], direct=[], max_phase=2),      # P1 (both teachers, A0) + P2 (all methods, seed 0)
    # full: final method set chosen by the tuning pilot (CWD + LSD 0.5 + CPD), trimmed grid (~50 runs)
    "full": dict(quick=False, method_set="v2",
                 first_kd=["B_logitKD", "B_AT", "B_CWD", "B_BPKD", "B_SARD", "B_Caug", "A5_LSD", "A6_CPD", "A7_full", "A8_CPDonly"],
                 #seed_kd=["B_AT", "B_CWD", "B_SARD", "B_Caug", "A5_LSD", "A6_CPD", "A7_full", "A8_CPDonly"],
                 seed_kd=["B_logitKD","B_AT","B_CWD","B_BPKD","B_SARD","B_Caug","A5_LSD","A6_CPD","A7_full","A8_CPDonly"],
                 ablations=[], sensitivity=["S_tau2", "S_lsd1", "S_cpd0.5", "S_cpd2", "S_clut0.75"],
                 gates=dict(DEFAULT_GATES, stop_on_fail=False)),   # decision already made by the tuning pilot
}
MODES["tune"] = dict(MODES["pilot"], tune=True, gates=dict(DEFAULT_GATES, stop_on_fail=False))
CFG.update(MODES[RUN_MODE], time_budget_h=TIME_BUDGET_H)
CFG.update(gpu_temp_high=82, gpu_temp_resume=76)   # lab GTX 1080 Ti crashed twice near 84-85 C: pause earlier
if RUN_MODE == "full" and FULL_TEACHER != "auto":
    CFG.update(teachers_to_train=[FULL_TEACHER], teacher_choice=FULL_TEACHER)
CACHE = CACHE.with_name(f"{CACHE.name}_{CFG['cache_long_side']}")       # one image cache per resolution
n_gpu = max(torch.cuda.device_count(), 1)
# training runs in a separate worker process (cell 5), so several data-loading processes work on Windows too;
# loaders created inside this notebook use workers=0 on Windows (Jupyter cannot spawn them reliably)
CFG["num_workers"] = max(1, min(6, (os.cpu_count() or 2) // n_gpu - 1))
NB_WORKERS = 0 if ON_WINDOWS else None
for k, v in OV.get("cfg", {}).items():
    CFG[k] = {**CFG[k], **v} if isinstance(v, dict) and isinstance(CFG.get(k), dict) else v
CFG["paths"] = dict(work=str(WORK), cache=str(CACHE), plantseg_root=str(PLANTSEG_ROOT), plantdoc_root=str(PLANTDOC_ROOT),
                    plantdoc_seg=str(PD_SEG) if PD_SEG else None)
if (WORK / "config.json").exists():                                 # settings must stay frozen across sessions
    old = L.load_cfg(WORK); skip = {"paths", "deadline", "num_workers", "time_budget_h", "gates", "ram_high", "ram_critical", "gpu_temp_high", "gpu_temp_resume"}
    diff = {k: (old.get(k), CFG.get(k)) for k in CFG if k not in skip and old.get(k) != CFG.get(k)}
    if diff:
        print("WARNING: settings differ from the previous session (results would not be comparable):", diff)
L.save_cfg(CFG, WORK)
(WORK / "requirements_frozen.txt").write_text(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)

print("Kaggle:", ON_KAGGLE, "| Colab:", IN_COLAB, "| Windows:", ON_WINDOWS, "| RUN_MODE:", RUN_MODE,
      "| img", CFG["img_size"], "| teachers", CFG["teachers_to_train"], "| students", CFG["students"] + CFG["secondary_students"])
print("PlantSeg root:", PLANTSEG_ROOT, "\nPlantDoc root:", PLANTDOC_ROOT, "\nPlantDoc-Seg masks:", PD_SEG, "\nWork dir:", WORK)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "none (CPU)")
if L.keep_awake():
    print("Windows sleep is blocked while this notebook runs (the screen may still turn off or lock).")
print("Python:", sys.executable, "| torch", torch.__version__, "| CUDA build", torch.version.cuda)
if not torch.cuda.is_available() and os.environ.get("P1_ALLOW_CPU") != "1":
    raise SystemExit("No GPU visible to PyTorch. This notebook is running in " + sys.executable +
                     " with torch " + torch.__version__ + ". Switch the Jupyter kernel to your GPU environment (cropai): "
                     "Kernel > Change kernel. If torch shows '+cpu' there too, reinstall the CUDA build (see the notebook notes).")
assert PLANTSEG_ROOT is not None, "PlantSeg not found: add the dataset (images/ + annotations/) as an input"

In [ ]:
# ===== 2. DATA: PAIRS, CACHE, LEAK-FREE 70/15/15 SPLIT, CLUTTER TEST SET  (methodology steps 4, 5, 8) =====
# First session: pairs every image with its mask by base name, reads mask statistics, groups near-duplicates
# (pHash Hamming <= 6, a group never spans two splits) and freezes a 70/15/15 split stratified by disease.
# Later sessions load the frozen split; only the image cache (resized to 768 px) is rebuilt.
import matplotlib.pyplot as plt, cv2
df = L.prepare_data(CFG, n_proc=os.cpu_count() or 2)
FR = L.load_split_frames(CFG)
print({k: len(v) for k, v in FR.items()}, "| diseases:", df.disease.nunique(), "| empty masks:", int((df.lesion_frac == 0).sum()))
print(df.groupby("split").agg(images=("uid", "size"), diseases=("disease", "nunique"), lesion_frac=("lesion_frac", "mean")).round(4))
BANK = L.make_bank(FR["train"], CFG)                 # occluders from TRAINING images only
CLUTTER_TEST = L.build_clutter_test(CFG, FR["test"], BANK)
print({k: f"{len(v)} images, mean altered share {v.altered.mean():.3f}" for k, v in CLUTTER_TEST.items()})

# check duplicate groups by eye (methodology: look at 30 groups before trusting the threshold)
g = df[df.group.duplicated(keep=False)].groupby("group")
show = list(g.groups.items())[:30]
if show:
    fig, ax = plt.subplots(len(show), 2, figsize=(4, 2 * len(show)), squeeze=False)
    for i, (gid, idx) in enumerate(show):
        for j, k in enumerate(list(idx)[:2]):
            r = df.loc[k]
            ax[i, j].imshow(cv2.cvtColor(cv2.imread(str(Path(CFG["paths"]["cache"]) / "img" / f"{r.uid:05d}.jpg")), cv2.COLOR_BGR2RGB))
            ax[i, j].set_title(f"{r.split} d{r.disease}", fontsize=7); ax[i, j].axis("off")
    plt.tight_layout(); (WORK / "figures").mkdir(exist_ok=True); plt.savefig(WORK / "figures" / "check_duplicate_groups.png", dpi=80); plt.close()
    print("saved figures/check_duplicate_groups.png")

In [ ]:
# ===== 3. VISUAL CHECKS: DATASET, PROXIES, CLUTTER  (methodology steps 6-8: check 20 samples by eye) =====
STD, MEAN = L.STD, L.MEAN
b = next(iter(L.make_loader(FR["train"], CFG, True, 4, workers=0)))
fig, ax = plt.subplots(4, 4, figsize=(10, 10))
for k in range(4):
    im = (b["img"][k].numpy().transpose(1, 2, 0) * STD + MEAN).clip(0, 1)
    for j, (t, x) in enumerate([("image", im), ("lesion mask", b["mask"][k]), ("size weight", b["size_w"][k]), ("leaf proxy", b["leaf"][k])]):
        ax[k, j].imshow(x); ax[k, j].set_title(t); ax[k, j].axis("off")
plt.tight_layout(); plt.savefig(WORK / "figures" / "check_dataset.png", dpi=90); plt.show()

fig, ax = plt.subplots(3, 4, figsize=(12, 9))
for i, key in enumerate(FR["test"].key[:3]):
    r = FR["test"][FR["test"].key == key].iloc[0]
    img, _ = L.read_pair(r); ax[i, 0].imshow(cv2.resize(img, (CFG["img_size"],) * 2)); ax[i, 0].set_title("clean")
    for j, lv in enumerate(["light", "medium", "heavy"]):
        f = CLUTTER_TEST[lv]; p = f[f.key == key].image.iloc[0]
        ax[i, j + 1].imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)); ax[i, j + 1].set_title(f"{lv} ({f[f.key == key].altered.iloc[0]:.0%} altered)")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(WORK / "figures" / "check_clutter_levels.png", dpi=90); plt.show()

In [ ]:
# ===== 4. LOSS UNIT TESTS  (methodology step 12: before any long run) =====
print(L.loss_unit_tests(CFG))

In [ ]:
# ===== 5. TRAIN EVERYTHING: resumable job queue, one worker per GPU  (methodology steps 10, 11, 13, 14) =====
# Each worker takes the next unfinished run, trains it, evaluates it on every test set, and moves on.
# Gates (week 3 and week 7) are checked automatically; a failed gate stops the queue with an explanation.
# Progress is printed every few minutes; full logs are in paper1_runs/logs/.
from IPython.display import clear_output
CFG["deadline"] = SESSION_START + TIME_BUDGET_H * 3600
L.save_cfg(CFG, WORK)
shutil.rmtree(WORK / "claims", ignore_errors=True)          # claims left by a session that was stopped
(WORK / "logs").mkdir(exist_ok=True)
gpus = list(range(torch.cuda.device_count())) or [0]
procs = []
for g in gpus:
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g) if torch.cuda.is_available() else "", PYTHONIOENCODING="utf-8")
    procs.append(subprocess.Popen([sys.executable, "-u", "p1lib.py", "worker", str(WORK), str(g)], env=env,
                                  stdout=open(WORK / "logs" / f"worker{g}.log", "a", encoding="utf-8"), stderr=subprocess.STDOUT))

def progress():
    rows = []
    for title, gt, jobs in L.phases(CFG):
        jobs = [L.resolve(CFG, j) for j in jobs]
        rows.append(dict(phase=title, runs=len(jobs), trained=sum(L.run_done(CFG, j["name"]) for j in jobs),
                         evaluated=sum(L.eval_done(CFG, j["name"]) for j in jobs)))
    return pd.DataFrame(rows)

while any(p.poll() is None for p in procs):
    time.sleep(120 if not QUICK else 10)
    clear_output(wait=True)
    print(f"session time used {(time.time() - SESSION_START) / 3600:.2f} h of {TIME_BUDGET_H} h")
    st = L.sys_stats(); gu = ""
    try:
        gu = subprocess.run(["nvidia-smi", "--query-gpu=temperature.gpu,utilization.gpu,memory.used,memory.total", "--format=csv,noheader"],
                            capture_output=True, text=True).stdout.strip()
    except Exception: pass
    print(f"RAM {st['ram_pct']}% used (free {st['ram_free_gb']} GB) | CPU {st['cpu_pct']}% | GPU temp C, util, mem used, total: {gu}"
          + ("   <-- RAM nearly full: training slows itself down" if st["ram_pct"] >= CFG["ram_high"] else ""))
    print(progress().to_string(index=False))
    for g in gpus:
        lines = (WORK / "logs" / f"worker{g}.log").read_text(encoding="utf-8", errors="replace").splitlines()
        print(f"\n--- worker {g} ---\n" + "\n".join(l[:220] for l in lines[-4:]))
clear_output(wait=True)
print(progress().to_string(index=False))
for g in gpus:
    print(f"--- worker {g} (last lines) ---"); print("\n".join((WORK / "logs" / f"worker{g}.log").read_text(encoding="utf-8", errors="replace").splitlines()[-6:]))
for f in sorted(WORK.glob("GATE_*.txt")):
    print(f.name, ":", f.read_text(encoding="utf-8", errors="replace").strip())
errs = list((WORK / "runs").glob("*/error.txt"))
if errs:
    print("ERRORS in:", [e.parent.name for e in errs]); print(errs[0].read_text(encoding="utf-8", errors="replace")[-3000:])
ALL_DONE = all(r.trained == r.runs and r.evaluated == r.runs for r in progress().itertuples())
print("FINISHED - all runs trained and evaluated" if ALL_DONE else
      "NOT FINISHED - start a new session (add this version's output as input) to continue")

In [ ]:
# ===== 6. VALIDATION SCORES + GATES =====
V = L.val_scores(CFG)
if len(V):
    V["best_val_miou"] *= 100
    print(V.pivot_table(index=["model", "method"], values="best_val_miou", aggfunc=["mean", "std", "count"]).round(2).to_string())
    print("\ntraining hours used so far:", round(V.train_hours.sum(), 1))
for f in sorted(WORK.glob("GATE_*.txt")):
    print(f.name, ":", f.read_text(encoding="utf-8", errors="replace").strip())
# teacher comparison (pilot): val mIoU and training time of each candidate; "auto" uses the higher val mIoU
rows = []
for k in CFG["teachers_to_train"]:
    p = WORK / "runs" / L.teacher_run(CFG, k) / "done.json"
    if p.exists():
        d = json.load(open(p)); rows.append(dict(teacher=k, val_miou=round(d["best_val_miou"] * 100, 2), train_hours=d["train_hours"],
                                                 epochs_run=d.get("epochs_run"), params_M=d["params_M"]))
if rows:
    print("\nTEACHERS:"); print(pd.DataFrame(rows).to_string(index=False))
    print("teacher used for distillation:", L.chosen_teacher(CFG),
          "\nRule of thumb: if SegFormer-B4 is < 3 mIoU better than ResNet-50, use resnet50 for the full run (much faster).")

In [ ]:
# ===== 7. RESULT TABLES: mean ± std over seeds, size bins, clutter, natural clutter, per disease, bootstrap  (step 17) =====
from scipy.stats import wilcoxon
RES = WORK / "results"; PI = RES / "per_image"
dev0 = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
TS = L.test_sets(CFG, FR)
for p in sorted((WORK / "runs").glob("*/done.json")):          # evaluate anything missing (e.g. PlantDoc masks added later)
    n = p.parent.name
    if any(not (PI / f"{n}__{t}.csv").exists() for t in TS):
        L.eval_run(CFG, n, FR, dev0, workers=NB_WORKERS)

def summarize(d):
    s = lambda c: d[c].sum()
    r = dict(miou=(s("inter1") / max(s("union1"), 1) + s("inter0") / max(s("union0"), 1)) / 2, iou_lesion=s("inter1") / max(s("union1"), 1),
             dice=d.dice.mean(), lesion_f1=2 * s("tp") / max(2 * s("tp") + s("fp") + s("fn"), 1), bf1=d.bf1.mean(), sev_mae=d.sev_err.mean())
    for b in ["small", "medium", "large"]:
        r[f"recall_{b}"] = s(f"tp_{b}") / max(s(f"tp_{b}") + s(f"fn_{b}"), 1)
    return r

rows, PER = [], {}
for p in sorted(PI.glob("*.csv")):
    run, test = p.stem.rsplit("__", 1); model, method, seed = L.parse_run(run)
    d = pd.read_csv(p, dtype={"key": str}); PER[(run, test)] = d
    rows.append(dict(run=run, model=model, method=method, seed=seed, test=test, n=len(d), **summarize(d)))
S = pd.DataFrame(rows); S.to_csv(RES / "test_summary.csv", index=False)
MET = ["miou", "iou_lesion", "dice", "lesion_f1", "recall_small", "recall_medium", "recall_large", "bf1", "sev_mae"]
ORDER = ["teacher", "direct", "A0_student"] + L.BASELINES + L.ABLATIONS + ["A5_LSD", "A6_CPD", "A8_CPDonly", "A7_full"] + list(L.SENSITIVITY)
def ms(g, cols):
    return pd.Series({c: f"{g[c].mean() * (1 if c == 'sev_mae' else 100):.2f} ± {g[c].std(ddof=0) * (1 if c == 'sev_mae' else 100):.2f}" for c in cols} | {"seeds": len(g)})
def table(test, methods=None, cols=MET):
    x = S[S.test == test]
    if methods is not None: x = x[x.method.isin(methods)]
    if x.empty: return pd.DataFrame()
    gb = x.groupby(["model", "method"])
    try:
        t = gb.apply(lambda g: ms(g, cols), include_groups=False).reset_index()
    except TypeError:                                           # pandas < 2.2
        t = gb.apply(lambda g: ms(g, cols)).reset_index()
    t["o"] = t.method.map({m: i for i, m in enumerate(ORDER)}); return t.sort_values(["model", "o"]).drop(columns="o")

MAIN = table("plantseg_test", [m for m in ORDER if not m.startswith("S_")]); MAIN.to_csv(RES / "table_main.csv", index=False)
print("MAIN TABLE - PlantSeg test (mIoU etc. x100, severity MAE in % of leaf area)"); display(MAIN)
ABL = table("plantseg_test", ["A0_student"] + L.ABLATIONS + ["A5_LSD", "A6_CPD", "A7_full"], ["miou", "iou_lesion", "lesion_f1", "recall_small", "bf1"])
ABL.to_csv(RES / "table_ablation.csv", index=False); print("ABLATION A0-A7"); display(ABL)

rob = S[~S.method.str.startswith("S_")].pivot_table(index=["model", "method"], columns="test", values="miou", aggfunc="mean") * 100
for lv in ["light", "medium", "heavy"]:
    if f"clutter_{lv}" in rob and "plantseg_test" in rob: rob[f"drop_{lv}"] = rob["plantseg_test"] - rob[f"clutter_{lv}"]
rob.round(2).to_csv(RES / "table_clutter.csv"); print("CLUTTER ROBUSTNESS (mIoU and drop from clean)"); display(rob.round(2))

SIZE = S[S.test.isin(["plantseg_test", "clutter_heavy"])].groupby(["test", "model", "method"])[["recall_small", "recall_medium", "recall_large", "lesion_f1"]].mean().mul(100).round(2)
SIZE.to_csv(RES / "table_lesion_size.csv"); print("LESION-LEVEL RECALL BY LESION SIZE (component < 0.5% / 0.5-5% / > 5% of image)"); display(SIZE)

# natural clutter bins: background share (1 - leaf proxy) terciles of the test split, cut-offs fixed from the test images
bg = FR["test"].bg_share; cuts = np.quantile(bg, [1 / 3, 2 / 3])
nat = []
for (run, test), d in PER.items():
    if test != "plantseg_test" or "bg_share" not in d: continue
    model, method, seed = L.parse_run(run)
    for name, m in [("low", d.bg_share < cuts[0]), ("mid", (d.bg_share >= cuts[0]) & (d.bg_share < cuts[1])), ("high", d.bg_share >= cuts[1])]:
        nat.append(dict(model=model, method=method, seed=seed, bg_bin=name, miou=summarize(d[m])["miou"]))
NAT = pd.DataFrame(nat)
if len(NAT):
    NAT = NAT.pivot_table(index=["model", "method"], columns="bg_bin", values="miou", aggfunc="mean")[["low", "mid", "high"]].mul(100).round(2)
    NAT.to_csv(RES / "table_natural_clutter.csv"); print(f"NATURAL CLUTTER: mIoU by background share (cut-offs {cuts.round(3)})"); display(NAT)

# per disease: top-20 most frequent diseases in training (stable per-class numbers)
top = FR["train"].disease.value_counts().index[:20]
pdz = []
for (run, test), d in PER.items():
    if test != "plantseg_test": continue
    model, method, seed = L.parse_run(run)
    for dz, g in d[d.disease.isin(top)].groupby("disease"):
        pdz.append(dict(model=model, method=method, seed=seed, disease=dz, miou=summarize(g)["miou"]))
PDZ = pd.DataFrame(pdz)
if len(PDZ):
    PDZ = PDZ.pivot_table(index="disease", columns=["model", "method"], values="miou", aggfunc="mean").mul(100).round(2)
    PDZ.to_csv(RES / "table_per_disease_top20.csv"); print("PER-DISEASE mIoU, top-20 diseases (columns: model, method)"); display(PDZ.iloc[:, :12])

# sensitivity (validation mIoU, methodology 5.3)
SENS = V[V.method.isin(["A7_full"] + list(L.SENSITIVITY)) & (V.seed == CFG["seeds"][0])] if len(V) else pd.DataFrame()
if len(SENS):
    SENS = SENS[SENS.model == L.short(CFG["students"][0])][["method", "best_val_miou"]].sort_values("method")
    SENS.to_csv(RES / "table_sensitivity_val.csv", index=False); print("SENSITIVITY (val mIoU x100, A7 variants)"); display(SENS)

# paired bootstrap (1,000 resamples of test images) + Wilcoxon: A7 vs best baseline (chosen on validation)
boot = []
for st in CFG["students"] + CFG.get("secondary_students", []):
    s_ = L.short(st); bb = L.best_baseline(CFG, s_) or L.best_baseline(CFG, L.short(CFG["students"][0]))
    if bb is None: continue
    for test in ["plantseg_test", "clutter_heavy"]:
        def per_img(method):
            ds = [d.set_index("key").iou for (run, t), d in PER.items() if t == test and run.startswith(f"{s_}__{method}__")]
            return pd.concat(ds, axis=1).mean(axis=1) if ds else None
        a, b_ = per_img("A7_full"), per_img(bb)
        if a is None or b_ is None: continue
        d = (a - b_.loc[a.index]).to_numpy(); rng = np.random.default_rng(0)
        bs = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(1000)]
        p = wilcoxon(d).pvalue if np.any(d != 0) else 1.0
        boot.append(dict(student=s_, test=test, best_baseline=bb, mean_gain=d.mean() * 100, ci_low=np.percentile(bs, 2.5) * 100,
                         ci_high=np.percentile(bs, 97.5) * 100, wilcoxon_p=p, n_images=len(d)))
BOOT = pd.DataFrame(boot)
if len(BOOT):
    BOOT.round(4).to_csv(RES / "table_bootstrap.csv", index=False); print("A7 vs BEST BASELINE: per-image lesion IoU gain (x100), 95% CI"); display(BOOT.round(3))

In [ ]:
# ===== 7b. TUNING SUMMARY (RUN_MODE = "tune"): which A7 variant to take into the full run =====
if CFG.get("tune"):
    st = L.short(CFG["students"][0]); meths = L.TUNE_BASE + L.TUNE_VARIANTS + L.TUNE_VARIANTS2
    v = V[(V.model == st) & V.method.isin(meths)].groupby("method").best_val_miou.agg(["mean", "std", "count"])
    t = (S[(S.model == st) & S.method.isin(meths) & S.test.isin(["plantseg_test", "clutter_medium", "clutter_heavy"])]
         .pivot_table(index="method", columns="test", values="miou", aggfunc=["mean", "std"]) * 100)
    t.columns = [f"{test}_{agg}" for agg, test in t.columns]
    T = pd.concat([v.add_prefix("val_"), t], axis=1).reindex(meths).round(2)
    T.to_csv(RES / "table_tuning.csv"); print("TUNING: mIoU x100, mean and std over seeds (val, clean test, medium and heavy clutter)"); display(T)
    T = T.dropna(how="all")
    bl = [m for m in T.index if m in L.BASELINES and m != "B_Caug"]
    bb = T.loc[bl, "val_mean"].idxmax(); caug = T.loc["B_Caug"]
    ours = [m for m in T.index if m == "A7_full" or m.startswith(("S_", "X_"))]
    best = T.loc[ours, "val_mean"].idxmax(); b = T.loc[best]
    print(f"\nbest of ours on validation: {best} ({b.val_mean:.2f}) vs best baseline {bb} ({T.loc[bb, 'val_mean']:.2f})")
    for c, lab in [("plantseg_test_mean", "clean test"), ("clutter_medium_mean", "medium clutter"), ("clutter_heavy_mean", "heavy clutter")]:
        print(f"  {lab:15s} {best}: {b[c]:.2f} | {bb}: {T.loc[bb, c]:.2f} ({b[c] - T.loc[bb, c]:+.2f}) | C-aug: {caug[c]:.2f} ({b[c] - caug[c]:+.2f})")

In [ ]:
# ===== 7c. PER-CLASS PERFORMANCE: every disease (115) and every crop  =====
# Uses the per-image test results already saved (no re-training). Class = disease ID stored in the mask pixels.
# Names (crop / disease) are read from the dataset's metadata CSV if one exists; otherwise IDs are shown.
import re as _re

def _class_names(root):
    """disease ID -> (crop, disease name), from any CSV/XLSX in the PlantSeg folder that lists image names with plant/disease."""
    keys = pd.concat([FR[s][["key", "disease"]] for s in ("train", "val", "test")])
    keyset = set(keys.key.astype(str))
    files = []
    for base in {Path(root), Path(root).parent, Path(root).parent.parent}:
        files += list(base.glob("*.csv")) + list(base.glob("*.xlsx"))
    for f in files:
        try:
            m = pd.read_csv(f, dtype=str) if f.suffix == ".csv" else pd.read_excel(f, dtype=str)
        except Exception:
            continue
        low = {c: c.lower() for c in m.columns}
        pc = next((c for c in m.columns if _re.search(r"plant|crop|host", low[c])), None)
        dc = next((c for c in m.columns if _re.search(r"disease|class|label", low[c]) and c != pc
                   and not _re.search(r"file|path|url", low[c])), None)
        kc = None
        for c in m.columns:                                   # the column holding image names
            st = m[c].astype(str).map(lambda v: Path(v.replace("\\", "/")).stem)
            if st.isin(keyset).mean() > 0.3:
                kc = c; m["_key"] = st; break
        if kc is None or (pc is None and dc is None):
            continue
        j = keys.astype({"key": str}).merge(m, left_on="key", right_on="_key")
        out = {}
        for dz, g in j.groupby("disease"):
            crop = g[pc].mode().iat[0] if pc else ""
            name = g[dc].mode().iat[0] if dc else ""
            out[int(dz)] = (str(crop).strip(), str(name).strip())
        print(f"class names from {f.name} (image column '{kc}', crop '{pc}', disease '{dc}'): {len(out)} classes")
        return out
    print("No metadata CSV with crop/disease names found next to the dataset - showing disease IDs only.\n"
          "Put PlantSeg's metadata CSV (image name + plant + disease columns) in", root, "and rerun this cell.")
    return {}

NAMES = _class_names(CFG["paths"]["plantseg_root"])

def _cls_metrics(g):
    s = lambda c: g[c].sum()
    i1 = s("inter1") / max(s("union1"), 1); i0 = s("inter0") / max(s("union0"), 1)
    return dict(n=len(g), miou=(i1 + i0) / 2 * 100, iou_lesion=i1 * 100, dice=g.dice.mean() * 100,
                lesion_f1=2 * s("tp") / max(2 * s("tp") + s("fp") + s("fn"), 1) * 100, sev_mae=g.sev_err.mean())

pc_rows = []
for (run, test), d in PER.items():
    if test != "plantseg_test":
        continue
    model, method, seed = L.parse_run(run)
    for dz, g in d.groupby("disease"):
        pc_rows.append(dict(model=model, method=method, seed=seed, disease=int(dz), **_cls_metrics(g)))
PC = pd.DataFrame(pc_rows)

if len(PC):
    PC["crop"] = PC.disease.map(lambda k: NAMES.get(k, ("", ""))[0])
    PC["disease_name"] = PC.disease.map(lambda k: NAMES.get(k, ("", f"class {k}"))[1])
    MCOLS = ["miou", "iou_lesion", "dice", "lesion_f1", "sev_mae"]
    # long table: every class x every method, mean over seeds (std when >1 seed)
    PCL = PC.groupby(["disease", "crop", "disease_name", "model", "method"]).agg(
        n_test=("n", "first"), seeds=("seed", "nunique"),
        **{f"{c}": (c, "mean") for c in MCOLS}, **{f"{c}_std": (c, "std") for c in MCOLS}).reset_index().round(2)
    PCL.to_csv(RES / "table_per_class_all_methods.csv", index=False)

    # wide table: one row per class, mIoU of the key models side by side
    st_short = L.short(CFG["students"][0])
    vs = S[(S.test == "plantseg_test") & (S.model == st_short)]
    base_m = vs[vs.method.isin(L.BASELINES)].groupby("method").miou.mean()
    ours = vs[vs.method.isin(["A7_full"] + [m for m in vs.method.unique() if m.startswith(("S_", "X_"))])].groupby("method").miou.mean()
    pick = {"student alone": (st_short, "A0_student")}
    if len(base_m): pick[f"best baseline ({base_m.idxmax()})"] = (st_short, base_m.idxmax())
    if "A7_full" in ours: pick["A7_full"] = (st_short, "A7_full")
    if len(ours) and ours.idxmax() != "A7_full": pick[f"best A7 variant ({ours.idxmax()})"] = (st_short, ours.idxmax())
    tk = L.chosen_teacher(CFG)
    if tk: pick["teacher"] = (f"teacher-{tk}", "teacher")
    W = None
    for lab, (mo, me) in pick.items():
        x = PCL[(PCL.model == mo) & (PCL.method == me)].set_index(["disease", "crop", "disease_name"])
        if x.empty: continue
        col = x[["miou"]].rename(columns={"miou": lab})
        W = col if W is None else W.join(col, how="outer")
        if "n_test" not in W: W.insert(0, "n_test", x.n_test)
    W = W.reset_index().sort_values("disease")
    W.to_csv(RES / "table_per_class_miou.csv", index=False)
    pd.set_option("display.max_rows", 200)
    print(f"PER-CLASS mIoU on PlantSeg test ({W.disease.nunique()} diseases; x100, mean over seeds). "
          f"Classes with n_test < 5 are unreliable.")
    display(W)

    # per crop
    if PC.crop.ne("").any():
        CR = []
        for (run, test), d in PER.items():
            if test != "plantseg_test": continue
            model, method, seed = L.parse_run(run)
            d = d.assign(crop=d.disease.map(lambda k: NAMES.get(int(k), ("", ""))[0]))
            for cr, g in d.groupby("crop"):
                CR.append(dict(model=model, method=method, seed=seed, crop=cr, **_cls_metrics(g)))
        CR = pd.DataFrame(CR)
        CRW = CR.pivot_table(index="crop", columns="method", values="miou", aggfunc="mean").round(2)
        CRW.insert(0, "n_test", CR.groupby("crop").n.first())
        CRW = CRW[["n_test"] + [m for _, (mo, m) in pick.items() if m in CRW.columns]]
        CRW.to_csv(RES / "table_per_crop_miou.csv")
        print("PER-CROP mIoU (x100)"); display(CRW.sort_values("n_test", ascending=False))

    # where the method helps / hurts most (classes with >= 5 test images)
    a_lab = next((k for k in pick if k.startswith("best A7")), "A7_full" if "A7_full" in pick else None)
    b_lab = next((k for k in pick if k.startswith("best baseline")), None)
    if a_lab and b_lab:
        r = W[W.n_test >= 5].assign(gain=lambda t: t[a_lab] - t[b_lab]).sort_values("gain")
        print(f"\n{a_lab} minus {b_lab}, per class (n_test >= 5): {(r.gain > 0).sum()} classes better, {(r.gain < 0).sum()} worse")
        print("hardest 10 classes for", a_lab); display(W[W.n_test >= 5].nsmallest(10, a_lab))
        print("biggest gains"); display(r.tail(5)); print("biggest losses"); display(r.head(5))
    print("saved: results/table_per_class_all_methods.csv, table_per_class_miou.csv, table_per_crop_miou.csv")
else:
    print("no per-image test results yet")


In [ ]:
# ===== 8. FIGURES  (methodology 5.5: importance maps, qualitative, accuracy vs latency, mIoU vs clutter, failures) =====
FIG = WORK / "figures"; FIG.mkdir(exist_ok=True)
def done_run(model, method, seed=0):
    n = f"{model}__{method}__s{seed}"; return n if L.run_done(CFG, n) else None
ST = L.short(CFG["students"][0])

# Fig 2: importance map examples (boundary, size, confidence, combined)
if L.teacher_spec(CFG):
    T = L.load_teacher(CFG, dev0)
    b = next(iter(L.make_loader(FR["val"].sort_values("lesion_frac").iloc[::max(1, len(FR["val"]) // 4)].head(4), CFG, False, 4, shuffle=False, workers=0)))
    y = b["mask"].to(dev0); x = b["img"].to(dev0)
    with torch.no_grad(): t_out = T(x).float()
    sw = torch.stack([torch.from_numpy(L.size_weight_map(m.numpy().astype(np.uint8))) for m in b["mask"]]).to(dev0)
    kd = CFG["kd"]; W = L.importance_map(y, sw, b["leaf"].to(dev0), t_out, kd)
    parts = [("image", None), ("boundary", L.boundary_band(y, kd["boundary_k"])), ("small-lesion", sw),
             ("teacher conf.", torch.softmax(t_out, 1).max(1).values), ("combined W", W)]
    fig, ax = plt.subplots(len(y), 5, figsize=(15, 3 * len(y)), squeeze=False)
    for k in range(len(y)):
        for j, (t, m) in enumerate(parts):
            ax[k, j].imshow((b["img"][k].numpy().transpose(1, 2, 0) * STD + MEAN).clip(0, 1) if m is None else m[k].cpu(), cmap=None if m is None else "magma")
            ax[k, j].set_title(t); ax[k, j].axis("off")
    plt.tight_layout(); plt.savefig(FIG / "fig2_importance_maps.png", dpi=150); plt.show()

# Fig 3: qualitative grid on small lesions and heavy clutter: image, GT, student alone, best baseline, ours
def qual(frame, keys, runs, fname):
    sub = frame.set_index("key").loc[keys].reset_index()
    b = next(iter(L.make_loader(sub, CFG, False, len(sub), shuffle=False, workers=0)))
    preds = []
    for title, rn in runs:
        m, _ = L.load_model(CFG, rn, dev0)
        with torch.no_grad(): preds.append((title, m(b["img"].to(dev0)).argmax(1).cpu()))
    fig, ax = plt.subplots(len(sub), 2 + len(preds), figsize=(3 * (2 + len(preds)), 3 * len(sub)), squeeze=False)
    for k in range(len(sub)):
        ax[k, 0].imshow((b["img"][k].numpy().transpose(1, 2, 0) * STD + MEAN).clip(0, 1)); ax[k, 0].set_title("image")
        ax[k, 1].imshow(b["mask"][k]); ax[k, 1].set_title("ground truth")
        for j, (t, p) in enumerate(preds): ax[k, 2 + j].imshow(p[k]); ax[k, 2 + j].set_title(t)
    for a in ax.ravel(): a.axis("off")
    plt.tight_layout(); plt.savefig(FIG / fname, dpi=150); plt.show()
bb = L.best_baseline(CFG, ST)
runs = [(t, r) for t, r in [("student alone", done_run(ST, "A0_student")), (f"best baseline ({bb})", done_run(ST, bb) if bb else None),
                            ("ours (A7)", done_run(ST, "A7_full"))] if r]
if runs:
    small_keys = FR["test"][FR["test"].lesion_frac > 0].sort_values("lesion_frac").key.head(4).tolist()
    qual(FR["test"], small_keys, runs, "fig3_qualitative_small_lesions.png")
    if "clutter_heavy" in TS: qual(TS["clutter_heavy"], small_keys, runs, "fig3_qualitative_heavy_clutter.png")

# Fig 5: mIoU vs clutter level, one line per method
lv = ["plantseg_test", "clutter_light", "clutter_medium", "clutter_heavy"]
if len(S):
    ALL_ST = CFG["students"] + CFG.get("secondary_students", [])
    fig, axs = plt.subplots(1, len(ALL_ST), figsize=(7 * len(ALL_ST), 4.5), squeeze=False)
    for a, st in zip(axs[0], ALL_ST):
        x = S[(S.model == L.short(st)) & S.test.isin(lv) & ~S.method.str.startswith("S_")].groupby(["method", "test"]).miou.mean().unstack()
        for m in [m for m in ORDER if m in x.index]:
            a.plot(range(4), [x.loc[m].get(t, np.nan) * 100 for t in lv], marker="o", label=m, lw=2.5 if m == "A7_full" else 1)
        a.set_xticks(range(4)); a.set_xticklabels(["clean", "light", "medium", "heavy"]); a.set_title(L.short(st)); a.set_ylabel("mIoU"); a.grid(alpha=.3)
    axs[0, -1].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout(); plt.savefig(FIG / "fig5_miou_vs_clutter.png", dpi=150); plt.show()

# Fig 6: failure cases of the full model (lowest lesion IoU on the test split)
r7 = done_run(ST, "A7_full")
if r7 and (r7, "plantseg_test") in PER:
    worst = PER[(r7, "plantseg_test")].query("lesion_frac > 0").sort_values("iou").key.head(6).tolist()
    qual(FR["test"], worst, [("ours (A7)", r7)], "fig6_failure_cases.png")
print("Fig 1 (method overview) is a drawing: make it by hand (teacher on clean image, student on cluttered image, three loss arrows, importance-map inset).")

In [ ]:
# ===== 9. EFFICIENCY, ONNX, INT8 TFLITE, INT8 ACCURACY CHECK  (methodology steps 18-20; step 21 needs the phone) =====
pip("onnx", "onnxscript")
from thop import profile
EXP = WORK / "export"; EXP.mkdir(exist_ok=True)
def efficiency(arch, enc, size=CFG["img_size"], runs=50):
    m = L.build(arch, enc, pretrained=False).eval(); x = torch.randn(1, 3, size, size)
    macs, _ = profile(m, inputs=(x,), verbose=False); torch.set_num_threads(4)
    with torch.no_grad():
        for _ in range(10): m(x)
        t = []
        for _ in range(runs):
            t0 = time.perf_counter(); m(x); t.append((time.perf_counter() - t0) * 1000)
    return dict(arch=arch, encoder=enc, params_M=round(L.n_params(m), 2), GFLOPs=round(2 * macs / 1e9, 2), cpu_ms_median=round(float(np.median(t)), 1))
models = ([(CFG["teachers"][k]["arch"], CFG["teachers"][k]["encoder"]) for k in CFG["teachers_to_train"]]
          + [("Unet", s) for s in CFG["students"] + CFG.get("secondary_students", [])] + [(d["arch"], d["encoder"]) for d in CFG["direct"]])
EFF = pd.DataFrame([efficiency(a, e, runs=5 if QUICK else 50) for a, e in models]); EFF.to_csv(RES / "efficiency_pc.csv", index=False); display(EFF)

def export_onnx(run):
    m, info = L.load_model(CFG, run, torch.device("cpu")); out = EXP / f"{run}.onnx"
    x = torch.randn(1, 3, CFG["img_size"], CFG["img_size"])
    try:
        torch.onnx.export(m, x, str(out), input_names=["input"], output_names=["logits"], opset_version=17, dynamo=False)
    except TypeError:
        torch.onnx.export(m, x, str(out), input_names=["input"], output_names=["logits"], opset_version=17)
    return out
to_export = [r for r in [done_run(L.short(s), m) for s in CFG["students"] + CFG.get("secondary_students", []) for m in ["A7_full", "A0_student"]] +
             [f"{d['name']}__direct__s0" for d in CFG["direct"]] + [L.teacher_run(CFG, k) for k in CFG["teachers_to_train"]]
             if r and L.run_done(CFG, r)]
ONNX = {r: export_onnx(r) for r in to_export}
print("ONNX:", {r: f"{p.stat().st_size / 1e6:.1f} MB" for r, p in ONNX.items()})

# calibration images for int8: 200 TRAINING images, NHWC, RGB in [0, 1] (onnx2tf applies mean/std itself)
cal = np.stack([cv2.resize(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB), (CFG["img_size"],) * 2) / 255.0
                for p in FR["train"].sample(min(200, len(FR["train"])), random_state=0).cimg]).astype(np.float32)
np.save(EXP / "calib_nhwc_0to1.npy", cal)

INT8 = []
try:
    assert ALL_DONE or QUICK, "int8 conversion runs once every model is trained (final session)"
    pip("onnx2tf", "onnx-graphsurgeon", "sng4onnx", "psutil", "ai-edge-litert", "tf_keras")
    import tensorflow as tf
    for run in [r for r in ONNX if "A7_full" in r or "A0_student" in r]:
        od = EXP / f"tflite_{run}"
        exe = shutil.which("onnx2tf") or str(Path(sys.executable).parent / ("Scripts/onnx2tf.exe" if os.name == "nt" else "onnx2tf"))
        cmd = [exe, "-i", str(ONNX[run]), "-o", str(od), "-oiqt", "-cind", "input", str(EXP / "calib_nhwc_0to1.npy"),
               str([[[MEAN.tolist()]]]), str([[[STD.tolist()]]])]
        r = subprocess.run(cmd[:5] + ["-tb", "tf_converter"] + cmd[5:], capture_output=True, text=True)   # TF converter path
        if "unrecognized arguments" in r.stderr:                                                         # older onnx2tf
            r = subprocess.run(cmd, capture_output=True, text=True)
        tfl = sorted(od.glob("*_integer_quant.tflite")); tfl = [p for p in tfl if "full_integer" not in p.name]
        if not tfl:
            print("onnx2tf failed for", run, r.stdout[-1500:], r.stderr[-1500:]); continue
        it = tf.lite.Interpreter(model_path=str(tfl[0])); it.allocate_tensors()
        inp, out = it.get_input_details()[0], it.get_output_details()[0]
        inter, union = np.zeros(2), np.zeros(2)
        for _, row in FR["test"].iterrows():
            img, y = L.read_pair(row); s = CFG["img_size"]
            img = cv2.resize(img, (s, s), interpolation=cv2.INTER_LINEAR); y = cv2.resize(y, (s, s), interpolation=cv2.INTER_NEAREST)
            x = ((img / 255.0 - MEAN) / STD).astype(np.float32)[None]
            if inp["shape"][1] == 3: x = x.transpose(0, 3, 1, 2)
            it.set_tensor(inp["index"], x); it.invoke(); o = it.get_tensor(out["index"])[0]
            p = o.argmax(-1) if o.shape[-1] == 2 else o.argmax(0)
            for c in (0, 1): inter[c] += ((p == c) & (y == c)).sum(); union[c] += ((p == c) | (y == c)).sum()
        fp32 = S[(S.run == run) & (S.test == "plantseg_test")].miou.iloc[0] * 100
        i8 = (inter / np.maximum(union, 1)).mean() * 100
        INT8.append(dict(run=run, fp32_miou=round(fp32, 2), int8_miou=round(i8, 2), drop=round(fp32 - i8, 2),
                         int8_MB=round(tfl[0].stat().st_size / 1e6, 2), gate_week10="PASS" if fp32 - i8 < 1.5 else "FAIL (use QAT)"))
except Exception as e:
    print("int8 TFLite step skipped:", repr(e)[:500], "\nThe ONNX files and calib_nhwc_0to1.npy are in export/; run onnx2tf on the lab PC instead.")
if INT8:
    INT8 = pd.DataFrame(INT8); INT8.to_csv(RES / "table_int8.csv", index=False); display(INT8)

# Fig 4: accuracy vs latency (CPU latency on this machine; replace with phone latency from step 21)
if len(S):
    acc = S[S.test == "plantseg_test"].groupby(["model", "method"]).miou.mean().mul(100).reset_index()
    lat = dict(zip(EFF.encoder, EFF.cpu_ms_median)); enc_of = dict(zip(V.model, V.encoder))
    fig, a = plt.subplots(figsize=(6, 4.5))
    for _, r in acc[acc.method.isin(["teacher", "direct", "A0_student", "A7_full"])].iterrows():
        enc = enc_of.get(r.model)
        if enc in lat:
            a.scatter(lat[enc], r.miou, s=50); a.annotate(f"{r.model} {r.method}", (lat[enc], r.miou), fontsize=7)
    a.set_xlabel("CPU latency (ms, this machine) - replace with phone latency"); a.set_ylabel("PlantSeg test mIoU"); a.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(FIG / "fig4_accuracy_vs_latency_cpu.png", dpi=150); plt.show()
print("""Phone (step 21, on your laptop with the phone over USB):
adb push benchmark_model /data/local/tmp/ && adb shell chmod +x /data/local/tmp/benchmark_model
adb push <model>_integer_quant.tflite /data/local/tmp/
adb shell /data/local/tmp/benchmark_model --graph=/data/local/tmp/<model>_integer_quant.tflite --num_threads=4 --warmup_runs=50 --num_runs=200 --use_gpu=false --report_peak_memory_footprint=true
(repeat with --use_gpu=true; record median latency, peak memory, file size, phone model and Android version)""")

In [ ]:
# ===== 10. PLANTDOC: IMAGES TO MASK, ANNOTATOR AGREEMENT, CROSS-DATASET CHECK  (methodology 3.4, week 1-4) =====
import collections, random as _r
PDS = Path(CFG["paths"]["plantdoc_seg"]) if CFG["paths"]["plantdoc_seg"] else WORK / "plantdoc_seg"
OUT_PD = WORK / "plantdoc_seg"; (OUT_PD / "images").mkdir(parents=True, exist_ok=True)
pd_imgs = []
if PLANTDOC_ROOT:
    pd_imgs = [p for p in Path(PLANTDOC_ROOT).rglob("*") if p.suffix.lower() in L.IMG_EXT and "test" in [x.lower() for x in p.parts]]
print("PlantDoc test images found:", len(pd_imgs))
if pd_imgs and not any((OUT_PD / "images").iterdir()):
    by = collections.defaultdict(list)
    for p in pd_imgs: by[p.parent.name].append(p)
    rng = _r.Random(0); chosen = []
    while len(chosen) < 200 and any(by.values()):
        for c in list(by):
            if by[c] and len(chosen) < 200: chosen.append(by[c].pop(rng.randrange(len(by[c]))))
    for i, p in enumerate(chosen):
        shutil.copy(p, OUT_PD / "images" / f"pd_{i:03d}_{p.parent.name.replace(' ', '_')}{p.suffix.lower()}")
    shutil.make_archive(str(WORK / "plantdoc_to_annotate"), "zip", OUT_PD / "images")
    print(f"copied {len(chosen)} images (spread over classes) -> plantdoc_to_annotate.zip for two annotators "
          "(LabelMe/CVAT; save PNG masks, lesion = 255, same base name, folders masks_a1/ and masks_a2/)")

# annotator agreement (IoU between annotators); annotator 1 becomes the ground truth in masks/
a1, a2 = PDS / "masks_a1", PDS / "masks_a2"
if a1.exists() and a2.exists():
    ious = []
    for p in a1.glob("*.png"):
        q = a2 / p.name
        if q.exists():
            x, y = cv2.imread(str(p), 0) > 0, cv2.imread(str(q), 0) > 0
            ious.append((x & y).sum() / max((x | y).sum(), 1))
    print(f"annotator agreement: mean IoU {np.mean(ious):.3f} over {len(ious)} images"); pd.DataFrame(dict(iou=ious)).to_csv(RES / "plantdoc_annotator_agreement.csv", index=False)
if "plantdoc_seg" in TS:
    print("PlantDoc-Seg mIoU (cross-dataset):"); display(S[S.test == "plantdoc_seg"].groupby(["model", "method"]).miou.agg(["mean", "std"]).mul(100).round(2))
else:
    print("No PlantDoc masks yet: cross-dataset numbers appear automatically once masks/ exists (add it as an input and rerun).")

# qualitative look now (no masks needed): teacher and full student on 6 PlantDoc images
if pd_imgs and L.teacher_spec(CFG):
    ex = _r.Random(1).sample(pd_imgs, min(6, len(pd_imgs))); s = CFG["img_size"]
    xb = torch.stack([L.to_tensor(cv2.resize(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB), (s, s))) for p in ex])
    nets = [("teacher", L.load_teacher(CFG, dev0))] + ([("ours (A7)", L.load_model(CFG, r7, dev0)[0])] if r7 else [])
    fig, ax = plt.subplots(len(ex), 1 + len(nets), figsize=(3 * (1 + len(nets)), 3 * len(ex)), squeeze=False)
    with torch.no_grad(): outs = [(t, m(xb.to(dev0)).argmax(1).cpu()) for t, m in nets]
    for k in range(len(ex)):
        ax[k, 0].imshow((xb[k].numpy().transpose(1, 2, 0) * STD + MEAN).clip(0, 1))
        for j, (t, o) in enumerate(outs): ax[k, j + 1].imshow(o[k]); ax[k, j + 1].set_title(t)
    for a in ax.ravel(): a.axis("off")
    plt.tight_layout(); plt.savefig(FIG / "plantdoc_qualitative.png", dpi=150); plt.show()

In [ ]:
# ===== 11. EXTRA METRICS: pixel precision/recall/accuracy + confusion matrix, PR & ROC curves (AP, ROC-AUC), Grad-CAM / Grad-CAM++ =====
# Runs on the saved models (runs/*/best.pt): no retraining. Safe to rerun; curve results are cached in results/prob_curves/.
# Note for the paper: pixel accuracy is dominated by healthy pixels (lesions are a few % of each image), so report it only
# next to mIoU / Dice; for a small foreground the precision-recall curve and AP are more informative than ROC.
import torch.nn.functional as F
FIG = WORK / "figures"; FIG.mkdir(exist_ok=True); STD, MEAN = L.STD, L.MEAN
_auc = lambda x, y: float(getattr(np, 'trapezoid', None)(y, x) if hasattr(np, 'trapezoid') else np.trapz(y, x))

# ---------- (a) pixel-level confusion counts from the per-image results already saved ----------
def pixel_counts(d):
    N = d.inter0 + d.union1                                   # TN + (TP + FP + FN) = all pixels of the image
    tp = d.inter1; gt = (d.lesion_frac * N).round(); pr = (d.pred_frac * N).round()
    fp = (pr - tp).clip(lower=0); fn = (gt - tp).clip(lower=0); tn = d.inter0
    return tp.sum(), fp.sum(), fn.sum(), tn.sum()

rows = []
for (run, test), d in PER.items():
    model, method, seed = L.parse_run(run)
    tp, fp, fn, tn = pixel_counts(d)
    rows.append(dict(model=model, method=method, seed=seed, test=test, TP=tp, FP=fp, FN=fn, TN=tn,
                     precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1), specificity=tn / max(tn + fp, 1),
                     accuracy=(tp + tn) / max(tp + tn + fp + fn, 1), f1=2 * tp / max(2 * tp + fp + fn, 1)))
PIX = pd.DataFrame(rows); PIX.to_csv(RES / "pixel_metrics_per_run.csv", index=False)
PIXT = (PIX[PIX.test == "plantseg_test"].groupby(["model", "method"])[["precision", "recall", "specificity", "accuracy", "f1"]]
        .agg(["mean", "std"]).mul(100).round(2))
PIXT.to_csv(RES / "table_pixel_metrics.csv"); print("PIXEL-LEVEL METRICS, PlantSeg test (x100, mean and std over seeds)"); display(PIXT)

def key_runs(seed=0):
    """teacher, student alone, best baseline, ours (A7) for the primary student, one seed."""
    st = L.short(CFG["students"][0]); bb = L.best_baseline(CFG, st); spec = L.teacher_spec(CFG)
    cand = [("teacher", spec["run"] if spec else None), ("student alone", f"{st}__A0_student__s{seed}"),
            (f"best baseline ({bb})", f"{st}__{bb}__s{seed}" if bb else None), ("ours (A7)", f"{st}__A7_full__s{seed}")]
    return [(t, r) for t, r in cand if r and L.run_done(CFG, r)]

# confusion matrices (row-normalised, summed over seeds of each method) for the key methods
KR = key_runs()
if KR:
    fig, axs = plt.subplots(1, len(KR), figsize=(3.6 * len(KR), 3.4), squeeze=False)
    for a, (title, run) in zip(axs[0], KR):
        model, method, _ = L.parse_run(run)
        x = PIX[(PIX.model == model) & (PIX.method == method) & (PIX.test == "plantseg_test")][["TN", "FP", "FN", "TP"]].sum()
        cm = np.array([[x.TN, x.FP], [x.FN, x.TP]], float); cmn = cm / cm.sum(1, keepdims=True)
        a.imshow(cmn, cmap="Greens", vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                a.text(j, i, f"{cmn[i, j]:.3f}\n({cm[i, j] / 1e6:.2f} M px)", ha="center", va="center", fontsize=8,
                       color="white" if cmn[i, j] > 0.5 else "black")
        a.set_xticks([0, 1]); a.set_xticklabels(["pred healthy", "pred lesion"]); a.set_yticks([0, 1]); a.set_yticklabels(["true healthy", "true lesion"])
        a.set_title(title, fontsize=9)
    plt.tight_layout(); plt.savefig(FIG / "fig_confusion_matrices.png", dpi=150); plt.show()

# ---------- (b) precision-recall and ROC curves from pixel probabilities (one extra pass per model) ----------
NB = 1000                                                     # probability histogram bins (exact up to 0.001)
@torch.no_grad()
def prob_hist(model, frame):
    pos = np.zeros(NB, np.int64); neg = np.zeros(NB, np.int64)
    dl = L.make_loader(frame, CFG, False, CFG["eval_bs"], shuffle=False, workers=NB_WORKERS)
    for b in dl:
        p = torch.softmax(model(b["img"].to(dev0)).float(), 1)[:, 1]
        bins = (p * (NB - 1)).round().long().flatten().cpu().numpy(); y = b["mask"].flatten().numpy().astype(bool)
        pos += np.bincount(bins[y], minlength=NB); neg += np.bincount(bins[~y], minlength=NB)
    return pos, neg

def curves(pos, neg):
    tp = np.cumsum(pos[::-1]); fp = np.cumsum(neg[::-1])          # threshold sweeps from high to low probability
    P, Nn = pos.sum(), neg.sum()
    rec = tp / max(P, 1); prec = tp / np.maximum(tp + fp, 1); fpr = fp / max(Nn, 1)
    ap = float(np.sum(np.diff(np.concatenate([[0], rec])) * prec))   # step-wise average precision
    roc_auc = float(_auc(np.concatenate([[0], fpr]), np.concatenate([[0], rec])))
    return dict(rec=rec, prec=prec, fpr=fpr, ap=ap, roc_auc=roc_auc)

PC = RES / "prob_curves"; PC.mkdir(exist_ok=True)
CURVE_TESTS = [t for t in ["plantseg_test", "clutter_heavy", "plantdoc_seg"] if t in TS]
crow = []
for p in sorted((WORK / "runs").glob("*/done.json")):
    run = p.parent.name; model, method, seed = L.parse_run(run); m = None
    if method.startswith("S_"):
        continue                                              # sensitivity variants: validation numbers are enough
    for test in CURVE_TESTS:
        f = PC / f"{run}__{test}.npz"
        try:
            z = np.load(f); z["pos"], z["neg"]
        except Exception:                                       # missing or half-written (e.g. power cut): redo
            if m is None:
                m, _ = L.load_model(CFG, run, dev0)
            pos, neg = prob_hist(m, TS[test])
            with open(str(f) + ".tmp", "wb") as fh: np.savez(fh, pos=pos, neg=neg)
            os.replace(str(f) + ".tmp", f); z = np.load(f)
        c = curves(z["pos"], z["neg"])
        crow.append(dict(model=model, method=method, seed=seed, test=test, AP=c["ap"], ROC_AUC=c["roc_auc"]))
    del m
    if dev0.type == "cuda": torch.cuda.empty_cache()
CUR = pd.DataFrame(crow)
if len(CUR):
    CT = CUR.groupby(["test", "model", "method"])[["AP", "ROC_AUC"]].agg(["mean", "std"]).mul(100).round(2)
    CT.to_csv(RES / "table_ap_rocauc.csv"); print("AVERAGE PRECISION and ROC-AUC (pixel level, x100, mean and std over seeds)"); display(CT)

if KR:
    fig, axs = plt.subplots(2, len(CURVE_TESTS), figsize=(5 * len(CURVE_TESTS), 9), squeeze=False)
    for j, test in enumerate(CURVE_TESTS):
        for title, run in KR:
            f = PC / f"{run}__{test}.npz"
            if not f.exists(): continue
            z = np.load(f); c = curves(z["pos"], z["neg"])
            axs[0, j].plot(c["rec"], c["prec"], label=f"{title} (AP {c['ap'] * 100:.1f})", lw=2.5 if "A7" in title else 1.2)
            axs[1, j].plot(c["fpr"], c["rec"], label=f"{title} (AUC {c['roc_auc'] * 100:.1f})", lw=2.5 if "A7" in title else 1.2)
        axs[0, j].set(xlabel="recall", ylabel="precision", title=f"Precision-recall, {test}", xlim=(0, 1), ylim=(0, 1.02))
        axs[1, j].set(xlabel="false positive rate", ylabel="true positive rate", title=f"ROC, {test}", xlim=(0, 1), ylim=(0, 1.02))
        axs[1, j].plot([0, 1], [0, 1], "k:", lw=0.8)
        for a in axs[:, j]: a.grid(alpha=.3); a.legend(fontsize=7, loc="lower left" if a in axs[0] else "lower right")
    plt.tight_layout(); plt.savefig(FIG / "fig_pr_roc_curves.png", dpi=150); plt.show()

# ---------- (c) Seg-Grad-CAM and Grad-CAM++ (qualitative explanation figure) ----------
# For segmentation the target score is the summed lesion logit over the pixels the model predicts as lesion
# (Vinogradova et al., Seg-Grad-CAM); the map is taken at encoder stage `CAM_STAGE` (index into the feature pyramid:
# 4 = stride 16, 5 = stride 32). Grad-CAM++ uses the second/third-order weighting of Chattopadhay et al.
CAM_STAGE = 4
def seg_cams(model, x):
    model.eval(); x = x.clone().requires_grad_(False)
    with torch.enable_grad():
        out, feats = L.forward_with_feats(model, x)
        A = feats[CAM_STAGE]; A.retain_grad()
        pred = (out.argmax(1, keepdim=True) == 1).float()
        if pred.sum() == 0:                                   # no predicted lesion: explain the most lesion-like region
            pred = (torch.softmax(out, 1)[:, 1:2] >= torch.softmax(out, 1)[:, 1:2].amax(dim=(2, 3), keepdim=True) * 0.5).float()
        score = (out[:, 1:2] * pred).sum()
        model.zero_grad(set_to_none=True); score.backward()
    g, a = A.grad.detach(), A.detach()
    w = g.mean(dim=(2, 3), keepdim=True)                      # Grad-CAM weights
    cam = F.relu((w * a).sum(1, keepdim=True))
    g2, g3 = g ** 2, g ** 3                                   # Grad-CAM++ weights
    alpha = g2 / (2 * g2 + a.sum(dim=(2, 3), keepdim=True) * g3 + 1e-7)
    wpp = (alpha * F.relu(g)).sum(dim=(2, 3), keepdim=True)
    campp = F.relu((wpp * a).sum(1, keepdim=True))
    up = lambda c: F.interpolate(c, size=x.shape[-2:], mode="bilinear", align_corners=False)
    norm = lambda c: (c - c.amin(dim=(2, 3), keepdim=True)) / (c.amax(dim=(2, 3), keepdim=True) - c.amin(dim=(2, 3), keepdim=True) + 1e-7)
    return norm(up(cam))[:, 0].cpu(), norm(up(campp))[:, 0].cpu(), pred[:, 0].cpu()

if KR:
    keys = FR["test"][FR["test"].lesion_frac > 0.005].sample(min(4, len(FR["test"])), random_state=0).key.tolist()
    sets = [("clean", FR["test"])] + ([("heavy clutter", TS["clutter_heavy"])] if "clutter_heavy" in TS else [])
    models = [(t, L.load_model(CFG, r, dev0)[0]) for t, r in KR if "teacher" not in t]
    for set_name, frame in sets:
        sub = frame.set_index("key").loc[keys].reset_index()
        b = next(iter(L.make_loader(sub, CFG, False, len(sub), shuffle=False, workers=0)))
        x = b["img"].to(dev0); res = [(t, seg_cams(m, x)) for t, m in models]
        cols = 2 + 2 * len(res)
        fig, ax = plt.subplots(len(sub), cols, figsize=(2.6 * cols, 2.6 * len(sub)), squeeze=False)
        for k in range(len(sub)):
            im = (b["img"][k].numpy().transpose(1, 2, 0) * STD + MEAN).clip(0, 1)
            ax[k, 0].imshow(im); ax[k, 0].set_title("image", fontsize=8); ax[k, 1].imshow(b["mask"][k]); ax[k, 1].set_title("ground truth", fontsize=8)
            for j, (t, (cam, campp, _)) in enumerate(res):
                for jj, (nm, c) in enumerate([("Grad-CAM", cam), ("Grad-CAM++", campp)]):
                    a = ax[k, 2 + 2 * j + jj]; a.imshow(im); a.imshow(c[k], cmap="jet", alpha=0.45); a.set_title(f"{t}\n{nm}", fontsize=7)
        for a in ax.ravel(): a.axis("off")
        plt.tight_layout(); plt.savefig(FIG / f"fig_gradcam_{set_name.replace(' ', '_')}.png", dpi=150); plt.show()
    del models
print("saved: results/table_pixel_metrics.csv, results/table_ap_rocauc.csv, figures/fig_confusion_matrices.png, "
      "figures/fig_pr_roc_curves.png, figures/fig_gradcam_*.png")


In [ ]:
# ===== 12. PACKAGE OUTPUTS: tables, figures, splits, logs, exported models -> paper1_outputs.zip =====
pkg = WORK.parent / "paper1_outputs"
shutil.rmtree(pkg, ignore_errors=True); pkg.mkdir()
for sub in ["results", "figures", "splits", "logs"]:
    if (WORK / sub).exists(): shutil.copytree(WORK / sub, pkg / sub, ignore=shutil.ignore_patterns("per_image", "evaldone"))
if (WORK / "export").exists(): shutil.copytree(WORK / "export", pkg / "export", ignore=shutil.ignore_patterns(
    "*.npy", "*.pb", "variables", "assets", "*float16*", "*int16*", "*dynamic_range*", "*float32.tflite", "teacher*"))
for f in ["config.json", "requirements_frozen.txt"] + [p.name for p in WORK.glob("GATE_*.txt")]:
    if (WORK / f).exists(): shutil.copy(WORK / f, pkg / f)
shutil.make_archive(str(pkg), "zip", pkg)
print("wrote", str(pkg) + ".zip", f"({(Path(str(pkg) + '.zip')).stat().st_size / 1e6:.1f} MB)")
print("Status:", "FINISHED" if ALL_DONE else "NOT FINISHED - run the next session to continue")